# Migration DAG Regression Test Suite

Replaces the manual regression pass over `migration_dag_mapr_to_s3.py` (DAG 1,
`source_to_s3_migration`) and its downstream consumer `migration_dag_iceberg.py`
(DAG 2, `iceberg_migration`). It triggers real DAG runs against the sample test data
already seeded by the `dev-tools/mapr-edge-node/scripts/setup-*.sh` scripts, then compares
the resulting `migration_table_status` / `iceberg_migration_table_status` rows, source-side
row and partition counts, and the DistCp `-m` / `-bandwidth` values in the task logs against
a single expected-results table.

Re-run this notebook top to bottom after every DAG code change.

### Which DAG is DAG 2

`migration_dag_iceberg.py` (`dag_id='iceberg_migration'`) is the DAG that consumes DAG 1's
S3 output: it owns `iceberg_migration_table_status` and the `migrate_tables_to_iceberg` task
that the `[11/11]` instructions in `setup-test-data.sh` name for P-03, P-04 and P-05.
`migration_dag_parquet_hms.py` (`dag_id='parquet_hms_registration'`) has no status table of
its own and is not part of this chain.

### Where the expected results come from

| Suite | Source of truth |
|---|---|
| `DAG-SCENARIO` | `setup-test-data.sh` `[6/11]` — DAG MIGRATION SCENARIOS table |
| `HIVE-TYPE-DDL` | `setup-test-data.sh` `[6/11]` — HIVE_TYPE_TO_SPARK_DDL table, verified with the `struct_checks` regex from `[5/11]` |
| `EMPTY-PARQUET` | `setup-test-data.sh` `[6/11]` — EMPTY PARQUET SCHEMA table + the TC-33 schema check in `[5/11]` |
| `PART-TYPES` | `setup-test-data.sh` `[5/11]` — `part_types_db` (WF-318) partition-type check |
| `REGRESSION` | `setup-test-data.sh` `[10/11]` — the `checks` list (exact row and partition counts) |
| `REGRESSION-BASELINE` | `setup-test-data.sh` `[11/11]` — REGRESSION BASELINE exception list |
| `TRANSIENT-RETRY` | `setup-test-data.sh` `[11/11]` — T-01, T-03, T-05 |
| `PERMANENT-FAIL` | `setup-test-data.sh` `[11/11]` — P-01 .. P-07 |
| `DISTCP-SIZING` | `setup-distcp-test-data.sh` `[5/5]` — expected sizing table and `size_job()` |

### Prerequisites

- **JupyterHub PySpark kernel** with Iceberg + S3 access — required, not optional: `spark.sql`
  is the only way to read the Iceberg tracking tables that hold every expected `overall_status`.
  On a plain Python kernel the notebook still runs and still triggers the DAGs, but every
  tracking-table check reports `SKIPPED-NO-SPARK` and the summary refuses to call the run a
  pass.
- `EDGE_SSH_TARGET` reachable by `ssh` from this node (reaches the MapR/HDFS source side and
  its HiveServer2 — the same edge node `CLUSTER_SSH_CONN_ID` points the DAG at). Source-side
  row/partition/DESCRIBE checks and the P-01/P-02 HDFS fault injection run there.
- Airflow stable REST API (`/api/v1`) reachable and credentialed.
- Test data seeded: `setup-test-data.sh` and `setup-distcp-test-data.sh` already run on the
  edge node.

Only stdlib, `subprocess` and PySpark are used — the same dependency set as
`test_distcp_partition_filter_scoped_validation.ipynb`, which reaches the Airflow side with
`urllib.request` rather than `requests`.

---
## Step 0: Configuration

Every value comes from an environment variable with a default matching what the repo already
uses: DAG ids from the three DAG files, `migration_tracking` from
`migrator_utils/migrations/shared.py`, `CLUSTER_SSH_CONN_ID` and the DistCp knobs from
`env.shared.example`, and the beeline / warehouse strings from the setup scripts.

`env.shared.example` carries no Airflow REST variables, so `AIRFLOW_*` are introduced here.
Set `AIRFLOW_AUTH_TOKEN` to use bearer auth; otherwise basic auth with
`AIRFLOW_USERNAME` / `AIRFLOW_PASSWORD` is used.

The DistCp sizing knobs default to the scaled-down test values that
`setup-distcp-test-data.sh` documents (1 MiB per mapper, min 1, max 8, 800 MB/s aggregate),
**not** the shipped production defaults. They must match the Airflow Variables in effect for
the run, or every `DISTCP-SIZING` row will fail.

### Running one DAG only

`REGRESSION_RUN_DAG1` and `REGRESSION_RUN_DAG2` both default to `true`. Set either to
`false` to run just the other — do not delete the config lines, because `DAG2_ID` and friends
are referenced by the trigger, probe, verification and report cells, and removing them raises
`NameError` several cells later.

With `REGRESSION_RUN_DAG2=false`:

- DAG 2 is never triggered, and the Step 2b probe stops checking that it exists
- the P-03/P-04 Parquet corruption is skipped, because nothing would consume it
- only DAG 1's Excel is uploaded
- every `DAG2`-phase scenario reports `SKIPPED-BY-CONFIG`, never a pass

With `REGRESSION_RUN_DAG1=false` the `DAG1`, `FAULT` and `EXCEL` phases are skipped the same
way. Note DAG 2 consumes DAG 1's S3 output, so running DAG 2 alone only makes sense against
output a previous run left behind; the notebook says so and skips it unless you set
`REGRESSION_SKIP_PREFLIGHT=true`.

### No values are baked in

Every setting is an environment variable. The file carries no credentials, hostnames, bucket
names, tenant names or DAG ids specific to any environment — the DAG-id defaults are the
canonical ones from the three DAG files, and the beeline/warehouse defaults are the ones
`setup-test-data.sh` uses for the containerised edge node. There is deliberately no default
Airflow password; the config cell reports what is missing instead of guessing.

Two categories default to **off**, because this file is committed and someone will run it
against the wrong cluster eventually:

| Default off | Why |
|---|---|
| `REGRESSION_ENABLE_P01/02/03/04/06` | They destroy fixtures: P-01 does `hdfs dfs -rm -r` on a source table (not recoverable without re-seeding), P-02 chmods it, P-03/P-04 overwrite a Parquet file with random bytes, P-06 uploads garbage over the Excel path. |
| `REGRESSION_APPLY_AIRFLOW_VARIABLES` | It writes Airflow Variables, which apply to every run of every DAG that reads them, not just this notebook's. |

Disabled cases report `SKIPPED-BY-CONFIG` with the reason, and are never counted as passes.

### Excel configs — generated by Step 3b

Step 3b generates both Excel configs from the row tables in that cell, uploads them to S3, and
passes each as `excel_file_path` in the dag_run conf. Set `REGRESSION_GENERATE_EXCEL=false` to
supply your own instead, in which case these must point at existing workbooks:

| Setting | Feeds | DAG Param default if unset |
|---|---|---|
| `REGRESSION_DAG1_EXCEL_S3_PATH` | `source_to_s3_migration` | `s3a://config-bucket/migration.xlsx` |
| `REGRESSION_DAG2_EXCEL_S3_PATH` | `iceberg_migration` | `s3a://config-bucket/iceberg_migration.xlsx` |

Those defaults are placeholders. If either path is left unset the DAG silently runs against
the placeholder and migrates a completely different table set, so the preflight refuses to
trigger rather than let that happen.

The Excel contents are the spec: the `[6/11]` DAG MIGRATION SCENARIOS table in
`setup-test-data.sh` lists one Excel row per scenario (source db, table token, dest db,
partition filter), and the scenario table in Step 3 is keyed to those rows. An Excel missing
a row means that scenario reports `SKIPPED-NO-ROW`, not a pass.

### DAG owner

`REGRESSION_DAG_OWNER` sets the owner the migration runs under. There are two separate
layers, and only one of them is settable per run:

| Layer | What it affects | How it is set |
|---|---|---|
| **DAG-level owner** (parse time) | `dag.owners` → the `pod_mutation_hook` sets `HADOOP_USER_NAME` / `SPARK_USER` on the task pod. **This is the principal Ranger authorizes and Keycloak resolves groups for.** Also the owner column in the Airflow UI. | Airflow Variable `migration_dag_owner` **only**. Read at DAG parse time, so dag_run conf can never reach it. |
| **Per-run owner** | `config['owner']`, and `spark.sql.kyuubi.session.user` set via `spark.conf` inside `get_config` | Variable/env `migration_dag_owner`, else dag_run conf `dag_owner`, else conf `spark_user`, else `data-migration` |

**`REGRESSION_DAG_OWNER` only reaches the per-run layer.** It sends `dag_owner` in the
dag_run conf, which sets `config['owner']` and the kyuubi session user. It does **not**
change the pod's `HADOOP_USER_NAME`/`SPARK_USER`, because that is fixed when the pod is
created from `dag.owners`, long before `get_config` runs.

So if a task fails with a Ranger `AccessControlException` naming `user [data-migration]`, or
a Keycloak `UserNotFoundException: User not found: data-migration`, the conf route cannot fix
it. The DAG-level owner must be changed by setting the Airflow Variable:

```python
set_airflow_variable("migration_dag_owner", "<your-keycloak-user>")
```

then waiting for the next DAG parse. The value must be a real Keycloak user with Ranger
privileges on the tracking database — `data-migration` is a placeholder that exists in
neither. Note this is a global change: it applies to every run of both DAGs, and it also
overrides the per-run layer. Nothing calls `set_airflow_variable` automatically.

Step 2c prints both layers and which one wins.

The `ENABLE_*` flags gate the fault-injection cases. P-03 is off by default because it needs
a DAG 2 Excel row for `migration_db_s3.customers` in SNAPSHOT mode, while the shipped config
has that table as INPLACE — see Step 4b.

In [ ]:
import base64
import csv
import io
import json
import os
import re
import shlex
import subprocess
import time
import urllib.error
import urllib.request

AIRFLOW_API_BASE   = os.environ.get("AIRFLOW_API_BASE", "http://localhost:8080")
AIRFLOW_BASE_PATH  = os.environ.get("AIRFLOW_BASE_PATH", "")
AIRFLOW_USERNAME   = os.environ.get("AIRFLOW_USERNAME", "")
AIRFLOW_PASSWORD   = os.environ.get("AIRFLOW_PASSWORD", "")
AIRFLOW_AUTH_TOKEN = os.environ.get("AIRFLOW_AUTH_TOKEN", "")
AIRFLOW_HTTP_TIMEOUT = int(os.environ.get("AIRFLOW_HTTP_TIMEOUT", "120"))

DAG1_ID = os.environ.get("REGRESSION_DAG1_ID", "source_to_s3_migration")
DAG2_ID = os.environ.get("REGRESSION_DAG2_ID", "iceberg_migration")

DAG1_DISTCP_TASK_ID  = os.environ.get("REGRESSION_DAG1_DISTCP_TASK", "run_distcp_ssh")
DAG1_EXCEL_TASK_ID   = os.environ.get("REGRESSION_DAG1_EXCEL_TASK", "parse_excel")
DAG1_PREREQ_TASK_ID  = os.environ.get("REGRESSION_DAG1_PREREQ_TASK", "validate_prerequisites")
DAG2_MIGRATE_TASK_ID = os.environ.get("REGRESSION_DAG2_MIGRATE_TASK", "migrate_tables_to_iceberg")

TRACKING_DB       = os.environ.get("MIGRATION_TRACKING_DATABASE", "migration_tracking")
DAG1_STATUS_TABLE = f"{TRACKING_DB}.migration_table_status"
DAG2_STATUS_TABLE = f"{TRACKING_DB}.iceberg_migration_table_status"

CLUSTER_SSH_CONN_ID = os.environ.get("CLUSTER_SSH_CONN_ID", "cluster_edge_ssh")
EDGE_SSH_TARGET     = os.environ.get("EDGE_SSH_TARGET", "")
EDGE_SSH_OPTS       = os.environ.get("EDGE_SSH_OPTS", "-o BatchMode=yes -o StrictHostKeyChecking=no")
EDGE_BEELINE        = os.environ.get(
    "EDGE_BEELINE", "/opt/hive/bin/beeline -u jdbc:hive2://localhost:10000 --silent=true")
EDGE_WAREHOUSE      = os.environ.get("EDGE_WAREHOUSE", "hdfs://localhost:9000/user/hive/warehouse")
EDGE_SHELL_TIMEOUT  = int(os.environ.get("EDGE_SHELL_TIMEOUT", "900"))

BUCKET             = os.environ.get("S3_BUCKET", "")
TENANT_PREFIX      = os.environ.get("S3_TENANT_PREFIX", "")
MIGRATION_S3_ROOT  = os.environ.get("MIGRATION_DEFAULT_S3_BUCKET", "") or (
    f"s3a://{BUCKET.rstrip('/')}/{TENANT_PREFIX.strip('/')}"
    if BUCKET and TENANT_PREFIX else "")
MIGRATION_INCLUDE_DB_IN_PATH = os.environ.get("MIGRATION_INCLUDE_DB_IN_PATH", "true").lower() == "true"

GENERATE_EXCEL_CONFIGS = os.environ.get(
    "REGRESSION_GENERATE_EXCEL", "true").lower() == "true"
EXCEL_OUTPUT_PREFIX = os.environ.get("REGRESSION_EXCEL_PREFIX", "") or (
    f"{MIGRATION_S3_ROOT}/regression_suite" if MIGRATION_S3_ROOT else "")

DAG1_EXCEL_S3_PATH = os.environ.get("REGRESSION_DAG1_EXCEL_S3_PATH", "") or (
    f"{EXCEL_OUTPUT_PREFIX}/regression_dag1.xlsx"
    if GENERATE_EXCEL_CONFIGS and EXCEL_OUTPUT_PREFIX else "")
DAG2_EXCEL_S3_PATH = os.environ.get("REGRESSION_DAG2_EXCEL_S3_PATH", "") or (
    f"{EXCEL_OUTPUT_PREFIX}/regression_dag2.xlsx"
    if GENERATE_EXCEL_CONFIGS and EXCEL_OUTPUT_PREFIX else "")

DEST_DB_SUFFIX     = os.environ.get("REGRESSION_DEST_DB_SUFFIX", "_s3")


def s3_dest(database):
    """DAG 1 destination database name, and therefore DAG 2's source database.

    Defined here, in the config cell, so the scenario table and the Excel rows derive it
    from one place. They used to spell it independently — the scenario table with a
    hardcoded '_s3' and the Excel rows via this function — so changing
    REGRESSION_DEST_DB_SUFFIX made the coverage check report phantom gaps.
    """
    return f"{database}{DEST_DB_SUFFIX}"
ALT_BUCKET         = os.environ.get("REGRESSION_ALT_BUCKET", "")
CUSTOM_S3_ENDPOINT = os.environ.get("REGRESSION_CUSTOM_S3_ENDPOINT", "")
CORRUPT_EXCEL_S3_PATH    = os.environ.get("CORRUPT_EXCEL_S3_PATH", "") or (
    f"{EXCEL_OUTPUT_PREFIX}/corrupt_config.xlsx" if EXCEL_OUTPUT_PREFIX else "")

DISTCP_TARGET_BYTES_PER_MAPPER = int(os.environ.get("TARGET_BYTES_PER_MAPPER", "1048576"))
DISTCP_MIN_MAPPERS             = int(os.environ.get("MIN_MAPPERS", "1"))
DISTCP_MAX_MAPPERS             = int(os.environ.get("MAX_MAPPERS", "8"))
DISTCP_TARGET_AGGREGATE_MBPS   = int(os.environ.get("TARGET_AGGREGATE_MBPS", "800"))

POLL_INTERVAL_SECONDS   = int(os.environ.get("REGRESSION_POLL_INTERVAL", "30"))
DAG_RUN_TIMEOUT_SECONDS = int(os.environ.get("REGRESSION_DAG_TIMEOUT", "7200"))

ENABLE_P01 = os.environ.get("REGRESSION_ENABLE_P01", "false").lower() == "true"
ENABLE_P02 = os.environ.get("REGRESSION_ENABLE_P02", "false").lower() == "true"
ENABLE_P03 = os.environ.get("REGRESSION_ENABLE_P03", "false").lower() == "true"
ENABLE_P04 = os.environ.get("REGRESSION_ENABLE_P04", "false").lower() == "true"
ENABLE_P06 = os.environ.get("REGRESSION_ENABLE_P06", "false").lower() == "true"

REGRESSION_DAG_OWNER = os.environ.get("REGRESSION_DAG_OWNER", "")

DAG_DEPLOY_S3_PREFIX = os.environ.get("REGRESSION_DAG_DEPLOY_S3_PREFIX", "")
ENV_CONFIG_S3_DIR = os.environ.get("REGRESSION_ENV_CONFIG_S3_DIR", "") or (
    f"{DAG_DEPLOY_S3_PREFIX.rstrip('/')}/migrator_utils/migration_configs"
    if DAG_DEPLOY_S3_PREFIX else "")
UPLOAD_ENV_FILE = os.environ.get("REGRESSION_UPLOAD_ENV_FILE", "true").lower() == "true"
OVERWRITE_FOREIGN_ENV = os.environ.get(
    "REGRESSION_OVERWRITE_FOREIGN_ENV", "false").lower() == "true"

ENV_FILE_NAME   = os.environ.get("REGRESSION_ENV_FILE_NAME", "") or f"env.{DAG1_ID}"
ENV_FILE_DIR    = os.environ.get("REGRESSION_ENV_FILE_DIR", "")
TRACKING_LOCATION = os.environ.get("MIGRATION_TRACKING_LOCATION", "") or (
    f"{MIGRATION_S3_ROOT}/{TRACKING_DB}" if MIGRATION_S3_ROOT else "")
REPORT_LOCATION = os.environ.get("MIGRATION_REPORT_LOCATION", "") or (
    f"{MIGRATION_S3_ROOT}/migration_reports" if MIGRATION_S3_ROOT else "")
EMAIL_RECIPIENTS = os.environ.get("MIGRATION_EMAIL_RECIPIENTS", "")

RUN_DAG1 = os.environ.get("REGRESSION_RUN_DAG1", "true").lower() == "true"
RUN_DAG2 = os.environ.get("REGRESSION_RUN_DAG2", "true").lower() == "true"
ACTIVE_DAG_IDS = ([DAG1_ID] if RUN_DAG1 else []) + ([DAG2_ID] if RUN_DAG2 else [])

SKIP_PREFLIGHT = os.environ.get("REGRESSION_SKIP_PREFLIGHT", "false").lower() == "true"
APPLY_AIRFLOW_VARIABLES = os.environ.get(
    "REGRESSION_APPLY_AIRFLOW_VARIABLES", "false").lower() == "true"
DAG_REPARSE_TIMEOUT_SECONDS = int(os.environ.get("REGRESSION_REPARSE_TIMEOUT", "900"))

AIRFLOW_VARIABLE_OVERRIDES = {
    key: value for key, value in (
        ("migration_dag_owner", REGRESSION_DAG_OWNER),
        ("migration_tracking_database", TRACKING_DB),
    ) if value
}
PREVIOUS_AIRFLOW_VARIABLES = {}

RUN_TAG   = os.environ.get("REGRESSION_RUN_TAG", time.strftime("%Y%m%dT%H%M%S"))
RUN_IDS   = {}
RESULTS   = []

print(f"Airflow API      : {AIRFLOW_API_BASE}"
      + (f"  (base path {AIRFLOW_BASE_PATH})" if AIRFLOW_BASE_PATH else ""))
print(f"DAG 1            : {DAG1_ID}" + ("" if RUN_DAG1 else "   [SKIPPED: REGRESSION_RUN_DAG1=false]"))
print(f"DAG 2            : {DAG2_ID}" + ("" if RUN_DAG2 else "   [SKIPPED: REGRESSION_RUN_DAG2=false]"))
print(f"DAG 1 status tbl : {DAG1_STATUS_TABLE}")
print(f"DAG 2 status tbl : {DAG2_STATUS_TABLE}")
print(f"Edge SSH target  : {EDGE_SSH_TARGET or '(unset — source-side checks will report SKIPPED-NO-EDGE)'}")
print(f"SSH conn id      : {CLUSTER_SSH_CONN_ID}")
print(f"DistCp knobs     : target={DISTCP_TARGET_BYTES_PER_MAPPER} min={DISTCP_MIN_MAPPERS} "
      f"max={DISTCP_MAX_MAPPERS} aggregate={DISTCP_TARGET_AGGREGATE_MBPS} MB/s")
print(f"Fault injection  : P-01={ENABLE_P01} P-02={ENABLE_P02} P-03={ENABLE_P03} "
      f"P-04={ENABLE_P04} P-06={ENABLE_P06}")
print(f"DAG owner        : {REGRESSION_DAG_OWNER or '(unset — DAG falls back to data-migration)'}")
print(f"Generate Excel   : {GENERATE_EXCEL_CONFIGS}")
print(f"DAG 1 Excel      : {DAG1_EXCEL_S3_PATH or '(UNSET — DAG would use its placeholder default)'}")
print(f"DAG 2 Excel      : {DAG2_EXCEL_S3_PATH or '(UNSET — DAG would use its placeholder default)'}")
print(f"Env config dir   : {ENV_CONFIG_S3_DIR or '(unset — env files will not be uploaded)'}")
print(f"Upload env files : {UPLOAD_ENV_FILE}")
print(f"Apply Variables  : {APPLY_AIRFLOW_VARIABLES} -> {AIRFLOW_VARIABLE_OVERRIDES or '(none)'}")
print(f"Run tag          : {RUN_TAG}")

_config_warnings = []
if not (AIRFLOW_AUTH_TOKEN or (AIRFLOW_USERNAME and AIRFLOW_PASSWORD)):
    _config_warnings.append(
        "No Airflow credentials. Set AIRFLOW_USERNAME + AIRFLOW_PASSWORD, or "
        "AIRFLOW_AUTH_TOKEN. This notebook holds no defaults for them by design.")
if not MIGRATION_S3_ROOT:
    _config_warnings.append(
        "MIGRATION_S3_ROOT is empty. Set MIGRATION_DEFAULT_S3_BUCKET to an s3a:// URI, or "
        "set both S3_BUCKET and S3_TENANT_PREFIX. Step 2c can also read it from the "
        "Airflow Variable migration_default_s3_bucket.")
elif not MIGRATION_S3_ROOT.startswith(("s3a://", "s3://")):
    _config_warnings.append(
        f"MIGRATION_S3_ROOT={MIGRATION_S3_ROOT!r} has no s3a:// scheme. Spark resolves a "
        f"schemeless path against its default filesystem, which repeats the bucket name.")
if not ACTIVE_DAG_IDS:
    _config_warnings.append(
        "Both REGRESSION_RUN_DAG1 and REGRESSION_RUN_DAG2 are false — nothing will run.")
for _label, _value in (("MIGRATION_TRACKING_LOCATION", TRACKING_LOCATION),
                       ("MIGRATION_REPORT_LOCATION", REPORT_LOCATION)):
    if _value and not _value.startswith(("s3a://", "s3://")):
        _config_warnings.append(f"{_label}={_value!r} is not an s3a:// URI.")
    elif not _value:
        _config_warnings.append(
            f"{_label} is empty, so the DAG falls back to its placeholder default under "
            f"s3a://data-lake/ — a bucket that is almost certainly not yours, which fails "
            f"with a 301 permanent redirect.")
if UPLOAD_ENV_FILE and not ENV_CONFIG_S3_DIR:
    _config_warnings.append(
        "REGRESSION_UPLOAD_ENV_FILE is true but neither REGRESSION_DAG_DEPLOY_S3_PREFIX "
        "nor REGRESSION_ENV_CONFIG_S3_DIR is set, so the env files cannot be placed where "
        "the DAG reads them. Every value below will fall back to an Airflow Variable or "
        "the hardcoded default in shared.py.")
elif ENV_CONFIG_S3_DIR and not ENV_CONFIG_S3_DIR.startswith(("s3a://", "s3://")):
    _config_warnings.append(
        f"ENV_CONFIG_S3_DIR={ENV_CONFIG_S3_DIR!r} is not an s3a:// URI.")
if not EDGE_SSH_TARGET:
    _config_warnings.append(
        "EDGE_SSH_TARGET is empty, so every source-side check (row counts, partition "
        "counts, struct DDL, schema) reports SKIPPED-NO-EDGE and P-01/P-02 cannot run.")

if _config_warnings:
    print("\nConfiguration notes:")
    for _warning in _config_warnings:
        print(f"  - {_warning}")

---
## Step 1a: Edge-node access and sizing arithmetic (no Spark required)

Everything in this cell is plain stdlib plus `subprocess`, so it works on any kernel. It is
deliberately separated from the Spark bootstrap in Step 1b: a kernel without PySpark must
still be able to reach the edge node and compute sizing expectations, rather than losing every
helper in the cell to one import error.

Source tables live on the MapR/HDFS edge node and are not visible to a Spark session running
here. Their row counts, `SHOW PARTITIONS` and `DESCRIBE` output are read over SSH through the
edge node's beeline, which is the access pattern the setup scripts use (`$BEELINE -e "..."`);
`[5/11]` and `[10/11]` run the equivalent queries through the edge node's own `pyspark`.

`edge_sql` asks beeline for `csv2` output so the result parses deterministically.

In [ ]:
class EdgeUnavailable(RuntimeError):
    pass


class SparkUnavailable(RuntimeError):
    pass


def edge_shell(command, check=False):
    if not EDGE_SSH_TARGET:
        raise EdgeUnavailable("EDGE_SSH_TARGET is not set")
    argv = ["ssh"] + shlex.split(EDGE_SSH_OPTS) + [EDGE_SSH_TARGET, command]
    completed = subprocess.run(argv, capture_output=True, text=True, timeout=EDGE_SHELL_TIMEOUT)
    if check and completed.returncode != 0:
        raise RuntimeError(f"edge command failed ({completed.returncode}): {completed.stderr[:500]}")
    return completed


def edge_sql(query):
    """Run a query through the edge node's beeline and return a list of dicts."""
    completed = edge_shell(f"{EDGE_BEELINE} --outputformat=csv2 -e {shlex.quote(query)}")
    if completed.returncode != 0:
        raise RuntimeError(f"beeline failed: {(completed.stderr or completed.stdout)[:500]}")
    reader = csv.DictReader(io.StringIO(completed.stdout))
    return [dict(row) for row in reader]


def edge_row_count(table, where=None):
    query = f"SELECT COUNT(*) AS c FROM {table}"
    if where:
        query += f" WHERE {where}"
    rows = edge_sql(query)
    return int(rows[0]["c"])


def edge_partition_count(table):
    return len(edge_sql(f"SHOW PARTITIONS {table}"))


def edge_partition_names(table):
    rows = edge_sql(f"SHOW PARTITIONS {table}")
    key = list(rows[0].keys())[0] if rows else None
    return {row[key] for row in rows} if key else set()


def edge_describe(table):
    rows = edge_sql(f"DESCRIBE {table}")
    described = []
    for row in rows:
        values = list(row.values())
        name = (values[0] or "").strip() if values else ""
        dtype = (values[1] or "").strip() if len(values) > 1 else ""
        described.append((name, dtype))
    return described


def edge_hdfs_path(database, table):
    return f"{EDGE_WAREHOUSE}/{database}.db/{table}"


def size_distcp_job(size_bytes, file_count):
    """Python port of size_job() in setup-distcp-test-data.sh [5/5]."""
    mappers = -(-int(size_bytes) // DISTCP_TARGET_BYTES_PER_MAPPER)
    mappers = min(mappers, DISTCP_MAX_MAPPERS)
    mappers = max(mappers, DISTCP_MIN_MAPPERS)
    if file_count > 0 and mappers > file_count:
        mappers = file_count
    mappers = max(mappers, DISTCP_MIN_MAPPERS)
    bandwidth = max(DISTCP_TARGET_AGGREGATE_MBPS // mappers, 1)
    return mappers, bandwidth


EDGE_AVAILABLE = False
if EDGE_SSH_TARGET:
    probe = edge_shell("hdfs dfs -ls / >/dev/null 2>&1 && echo EDGE_OK")
    EDGE_AVAILABLE = "EDGE_OK" in probe.stdout
    print(f"Edge node reachable: {EDGE_AVAILABLE}")
else:
    print("Edge node not configured — source-side checks and P-01/P-02 will be reported, not faked.")

---
## Step 1b: Spark session and S3 helpers

**This notebook needs a PySpark kernel to verify anything.** The tracking tables
(`migration_table_status`, `iceberg_migration_table_status`) are Iceberg tables on S3, and
`spark.sql` is the only access path to them — the edge node's beeline points at the MapR Hive
metastore on the source side and cannot read them. Spark is also what writes the P-03/P-04
Parquet corruption and the P-06 Excel object, through the Hadoop `FileSystem` API off the live
session.

The session is reused if this kernel already has one, matching
`test_distcp_partition_filter_scoped_validation.ipynb`.

If PySpark is not importable the cell does **not** raise. It sets `SPARK_AVAILABLE = False`,
and every check that needs Spark reports `SKIPPED-NO-SPARK` in the final report instead of
passing. That keeps a wrong-kernel run visibly incomplete rather than silently green — but it
is not a usable mode for signing off a regression: the DAG runs still trigger and the
source-side checks still work, while every `overall_status` assertion is skipped.

On a JupyterHub node the fix is to select the PySpark kernel. Locally, `pip install pyspark`
gets the import to resolve but will not by itself give the session Iceberg or S3 credentials.

In [ ]:
SPARK_AVAILABLE = False
SPARK_IMPORT_ERROR = None

try:
    from pyspark.sql import SparkSession
    try:
        _ = spark
        print(f"Using existing Spark session (version {spark.version})")
    except NameError:
        spark = SparkSession.builder \
            .appName("migration-dag-regression-suite") \
            .enableHiveSupport() \
            .getOrCreate()
        print(f"Created Spark session (version {spark.version})")
    spark.sparkContext.setLogLevel("WARN")
    SPARK_AVAILABLE = True
except Exception as err:
    SPARK_IMPORT_ERROR = f"{type(err).__name__}: {err}"
    spark = None
    print(f"Spark unavailable — {SPARK_IMPORT_ERROR}")
    print("Tracking-table checks will report SKIPPED-NO-SPARK. Select a PySpark kernel to "
          "run the suite for real.")


def require_spark():
    if not SPARK_AVAILABLE:
        raise SparkUnavailable(SPARK_IMPORT_ERROR or "no Spark session in this kernel")
    return spark


def _fs(path):
    session = require_spark()
    return session._jvm.org.apache.hadoop.fs.FileSystem.get(
        session._jvm.java.net.URI(path), session._jsc.hadoopConfiguration()
    )


def _hadoop_path(path):
    return require_spark()._jvm.org.apache.hadoop.fs.Path(path)


def s3_exists(path):
    return _fs(path).exists(_hadoop_path(path))


def s3_delete(path):
    fs = _fs(path)
    p = _hadoop_path(path)
    if fs.exists(p):
        fs.delete(p, True)
        print(f"  Deleted: {path}")
        return True
    print(f"  Not found (skip): {path}")
    return False


def s3_list_files(path, suffix=None):
    fs = _fs(path)
    p = _hadoop_path(path)
    if not fs.exists(p):
        return []
    found = []
    walker = fs.listFiles(p, True)
    while walker.hasNext():
        entry = walker.next()
        name = entry.getPath().toString()
        if suffix is None or name.endswith(suffix):
            found.append(name)
    return sorted(found)


def s3_read_text(path):
    fs = _fs(path)
    p = _hadoop_path(path)
    if not fs.exists(p):
        return None
    stream = fs.open(p)
    try:
        chunk = bytearray()
        one = stream.read()
        while one != -1:
            chunk.append(one)
            if len(chunk) >= 4096:
                break
            one = stream.read()
    finally:
        stream.close()
    return bytes(chunk).decode("utf-8", "replace")


def s3_write_bytes(path, payload):
    fs = _fs(path)
    stream = fs.create(_hadoop_path(path), True)
    try:
        stream.write(bytearray(payload))
    finally:
        stream.close()
    print(f"  Wrote {len(payload)} bytes to {path}")


if SPARK_AVAILABLE:
    from py4j.java_gateway import java_import
    java_import(spark._jvm, "org.apache.hadoop.fs.*")

print(f"SPARK_AVAILABLE = {SPARK_AVAILABLE}")

---
## Step 2: Airflow REST API client (Airflow 2 and 3)

Thin `urllib.request` wrapper. Airflow 2 and Airflow 3 differ in both the API path and the
auth scheme, so the client handles both and Step 2b detects which one is actually there rather
than assuming:

| | Airflow 2.x | Airflow 3.x |
|---|---|---|
| Base path | `/api/v1` | `/api/v2` |
| Auth | HTTP basic | JWT bearer, token from `POST /auth/token` |
| Create-run body | `dag_run_id`, `conf` | `dag_run_id`, `conf`, `logical_date` |

`AIRFLOW_API_BASE` may be given either as a full base (`http://host:8080/api/v2`) or as just
the origin (`http://host:8080`) — Step 2b probes `/api/v2` then `/api/v1` and pins whichever
answers.

Four endpoints are used, identical in shape across both versions:

| Purpose | Endpoint |
|---|---|
| Trigger a run | `POST /dags/{dag_id}/dagRuns` |
| Poll run state | `GET /dags/{dag_id}/dagRuns/{dag_run_id}` |
| Read attempt counts | `GET /dags/{dag_id}/dagRuns/{dag_run_id}/taskInstances` |
| Read task logs | `GET /dags/{dag_id}/dagRuns/{dag_run_id}/taskInstances/{task_id}/logs/{try_number}` |

`wait_for_dag_run` sleeps `POLL_INTERVAL_SECONDS` between polls and gives up after
`DAG_RUN_TIMEOUT_SECONDS` — no busy-wait. A failed DAG run is a normal outcome here (the
permanent-fail passes are supposed to fail), so it is returned rather than raised.

On an HTTP error the raised message includes the full URL and, for `405`, the server's `Allow`
header — a `405` on `POST .../dagRuns` almost always means the base path belongs to a
different API version, or a read-only proxy sits in front of Airflow.

In [ ]:
TERMINAL_DAG_STATES = {"success", "failed"}

AIRFLOW_ROOT = None
AIRFLOW_API_VERSION = None
AIRFLOW_BASE_RESOLVED = None
AIRFLOW_BEARER_TOKEN = AIRFLOW_AUTH_TOKEN or None

DIAGNOSTIC_PATHS = ("/api/v2/version", "/api/v1/version", "/api/v2/monitor/health",
                    "/health", "/login", "/")


class _NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None


def _strip_api_suffix(url):
    trimmed = url.rstrip("/")
    for suffix in ("/api/v1", "/api/v2"):
        if trimmed.endswith(suffix):
            return trimmed[: -len(suffix)]
    return trimmed


def _basic_auth_header():
    raw = f"{AIRFLOW_USERNAME}:{AIRFLOW_PASSWORD}".encode()
    return "Basic " + base64.b64encode(raw).decode()


def _auth_header():
    if AIRFLOW_BEARER_TOKEN:
        return f"Bearer {AIRFLOW_BEARER_TOKEN}"
    return _basic_auth_header()


def _http(method, url, payload=None, as_text=False, authorization=None, timeout=None):
    body = json.dumps(payload).encode() if payload is not None else None
    request = urllib.request.Request(url, data=body, method=method)
    request.add_header("Content-Type", "application/json")
    request.add_header("Accept", "text/plain" if as_text else "application/json")
    header = authorization if authorization is not None else _auth_header()
    if header:
        request.add_header("Authorization", header)
    with urllib.request.urlopen(request, timeout=timeout or AIRFLOW_HTTP_TIMEOUT) as response:
        raw = response.read().decode("utf-8", "replace")
    if as_text:
        return raw
    return json.loads(raw) if raw.strip() else {}


def probe_url(url, method="GET"):
    """Single probe that reports status, content type, redirect target and a body snippet."""
    request = urllib.request.Request(url, method=method)
    request.add_header("Accept", "application/json")
    header = _auth_header()
    if header:
        request.add_header("Authorization", header)
    opener = urllib.request.build_opener(_NoRedirect)
    try:
        with opener.open(request, timeout=min(AIRFLOW_HTTP_TIMEOUT, 20)) as response:
            headers = response.headers
            return {"status": response.status, "content_type": headers.get("Content-Type", ""),
                    "location": headers.get("Location", ""),
                    "body": response.read(300).decode("utf-8", "replace")}
    except urllib.error.HTTPError as err:
        headers = err.headers
        return {"status": err.code,
                "content_type": headers.get("Content-Type", "") if headers else "",
                "location": headers.get("Location", "") if headers else "",
                "body": err.read(300).decode("utf-8", "replace")}
    except Exception as err:
        return {"status": None, "content_type": "", "location": "",
                "body": f"{type(err).__name__}: {err}"}


def _is_json(result):
    return "json" in (result.get("content_type") or "").lower()


def _served_by_airflow(result):
    """An Airflow API answer is JSON; an nginx/ingress miss is HTML."""
    return result["status"] is not None and (
        (result["status"] == 200 and _is_json(result))
        or result["status"] in (401, 403))


def candidate_roots():
    base = _strip_api_suffix(AIRFLOW_API_BASE).rstrip("/")
    roots = []

    def add(root):
        root = root.rstrip("/")
        if root and root not in roots:
            roots.append(root)

    for root in (base + AIRFLOW_BASE_PATH, base, base + "/airflow"):
        add(root)
        if root.startswith("http://"):
            add("https://" + root[len("http://"):])
    return roots


def explore_airflow_endpoints():
    """Print what each candidate root actually serves — run this when detection fails."""
    print("{:<52} {:<22} {:>6}  {}".format("URL", "CONTENT-TYPE", "CODE", "NOTE"))
    print("-" * 118)
    for root in candidate_roots():
        for path in DIAGNOSTIC_PATHS:
            result = probe_url(f"{root}{path}")
            note = ""
            if result["status"] is None:
                note = result["body"][:44]
            elif result["location"]:
                note = f"-> {result['location'][:44]}"
            elif _served_by_airflow(result):
                note = "ANSWERED BY AIRFLOW"
            elif "nginx" in result["body"].lower() or "<html" in result["body"].lower():
                note = "html error page (proxy/ingress, not Airflow)"
            print("{:<52} {:<22} {:>6}  {}".format(
                f"{root}{path}"[-52:], (result["content_type"] or "-")[:22],
                result["status"] if result["status"] is not None else "ERR", note))
    print()
    print("Looking for a row marked ANSWERED BY AIRFLOW. If every row is an html error page,")
    print("the hostname is not routed to Airflow at all — the URL is wrong, or the service")
    print("is not exposed there. Open the Airflow UI in a browser and copy the address bar")
    print("origin into AIRFLOW_API_BASE (plus AIRFLOW_BASE_PATH if the UI sits under a path).")


def fetch_jwt_token(root):
    """Airflow 3 auth: exchange username/password at /auth/token for a bearer token.

    Falls back to the credential-less GET form, which is what simple auth manager
    serves when [core] simple_auth_manager_all_admins is True.
    """
    try:
        payload = _http("POST", f"{root}/auth/token",
                        {"username": AIRFLOW_USERNAME, "password": AIRFLOW_PASSWORD},
                        authorization="")
    except urllib.error.HTTPError as err:
        if err.code not in (404, 405):
            raise
        payload = _http("GET", f"{root}/auth/token", authorization="")
    return payload.get("access_token")


def detect_airflow_api():
    """Pin AIRFLOW_BASE_RESOLVED / AIRFLOW_API_VERSION / auth mode by probing the server."""
    global AIRFLOW_ROOT, AIRFLOW_API_VERSION, AIRFLOW_BASE_RESOLVED, AIRFLOW_BEARER_TOKEN

    findings = []
    for root in candidate_roots():
        for version in ("v2", "v1"):
            url = f"{root}/api/{version}/version"
            result = probe_url(url)
            label = f"GET {url}"
            if _served_by_airflow(result):
                findings.append((label, result["status"], "answered by Airflow"))
                AIRFLOW_ROOT, AIRFLOW_API_VERSION = root, version
                AIRFLOW_BASE_RESOLVED = f"{root}/api/{version}"
                break
            detail = result["body"][:60].replace("\n", " ") if result["status"] is None else (
                "html error page" if not _is_json(result) else result["body"][:60])
            findings.append((label, result["status"] or "ERR", detail))
        if AIRFLOW_BASE_RESOLVED:
            break

    if AIRFLOW_BASE_RESOLVED and AIRFLOW_API_VERSION == "v2" and not AIRFLOW_BEARER_TOKEN:
        try:
            token = fetch_jwt_token(AIRFLOW_ROOT)
        except Exception as err:
            token = None
            findings.append(("POST /auth/token", "-", f"{type(err).__name__}: {err}"[:100]))
        if token:
            AIRFLOW_BEARER_TOKEN = token
            findings.append(("POST /auth/token", 200, "obtained JWT bearer token"))

    for label, code, detail in findings:
        print(f"  probe {label} -> {code}  {detail}")

    if not AIRFLOW_BASE_RESOLVED:
        print("\nNo candidate root answered like an Airflow API.")
        print("Running endpoint exploration to show what is actually being served...\n")
        explore_airflow_endpoints()
        return None

    print(f"\nResolved API base : {AIRFLOW_BASE_RESOLVED}")
    print(f"API version       : {AIRFLOW_API_VERSION} "
          f"({'Airflow 3.x' if AIRFLOW_API_VERSION == 'v2' else 'Airflow 2.x'})")
    print(f"Auth mode         : {'JWT bearer' if AIRFLOW_BEARER_TOKEN else 'HTTP basic'}")
    if AIRFLOW_API_VERSION == "v2" and not AIRFLOW_BEARER_TOKEN:
        print("  WARNING: Airflow 3 detected but no JWT was obtained. Set AIRFLOW_AUTH_TOKEN "
              "directly, or fix AIRFLOW_USERNAME / AIRFLOW_PASSWORD — triggering will fail.")
    return AIRFLOW_BASE_RESOLVED


def airflow_request(method, path, payload=None, as_text=False):
    if AIRFLOW_BASE_RESOLVED is None:
        raise RuntimeError(
            "Airflow API base was never resolved — run the Step 2b probe first. "
            "The last detection attempt found no endpoint that answers like Airflow.")
    base = AIRFLOW_BASE_RESOLVED
    url = f"{base}{path}"
    try:
        return _http(method, url, payload, as_text)
    except urllib.error.HTTPError as err:
        detail = err.read().decode("utf-8", "replace")[:800]
        hint = ""
        if err.code == 404 and ("<html" in detail.lower() or "nginx" in detail.lower()):
            hint = (" | this is an HTML error page from a proxy/ingress, not Airflow: the "
                    "host or base path does not route to Airflow. Run "
                    "explore_airflow_endpoints() to see what is served.")
        if err.code == 405:
            allowed = err.headers.get("Allow") if err.headers else None
            hint = (f" | server allows: {allowed}" if allowed else "")
            hint += (" | a 405 on this path usually means the API base belongs to a "
                     "different Airflow version, or a read-only proxy is in front of "
                     "Airflow. Re-run the Step 2b probe.")
        raise RuntimeError(f"{method} {url} -> HTTP {err.code}: {detail}{hint}") from None


S3_URI_SCHEMES = ("s3a://", "s3://")


def looks_like_s3_uri(path):
    return str(path or "").startswith(S3_URI_SCHEMES)


def suggest_s3_uri(path):
    """Best-effort corrected URI for a path written without a scheme.

    BUCKET may itself have been written with a scheme, so it is normalised to a bare
    bucket name before use; otherwise the suggestion comes back with s3a:// twice.
    """
    bare = str(path or "").lstrip("/")
    bucket = str(BUCKET or "")
    for scheme in S3_URI_SCHEMES:
        if bucket.startswith(scheme):
            bucket = bucket[len(scheme):]
    bucket = bucket.strip("/")
    if not bucket:
        return f"s3a://<bucket>/{bare}"
    if bare == bucket or bare.startswith(f"{bucket}/"):
        return f"s3a://{bare}"
    return f"s3a://{bucket}/{bare}"


def assert_ready_to_trigger(dag_id):
    """Refuse to trigger when a known-broken precondition would waste the run."""
    if dag_id not in ACTIVE_DAG_IDS:
        raise RuntimeError(
            f"{dag_id} is disabled for this run (REGRESSION_RUN_DAG1={RUN_DAG1}, "
            f"REGRESSION_RUN_DAG2={RUN_DAG2}). Its scenarios report SKIPPED-BY-CONFIG.")
    if SKIP_PREFLIGHT:
        return
    if not globals().get("AIRFLOW_REACHABLE", False):
        raise RuntimeError(
            "Step 2b has not confirmed the Airflow API. Run it before triggering.")
    if APPLY_AIRFLOW_VARIABLES and not globals().get("VARIABLES_READY", False):
        raise RuntimeError(
            "Step 2d did not confirm the Airflow Variables took effect, so this run would "
            "use the old parse-time owner and fail Ranger the same way. Run Step 2d, or "
            "set REGRESSION_SKIP_PREFLIGHT=true to override.")
    excel_path = DAG1_EXCEL_S3_PATH if dag_id == DAG1_ID else DAG2_EXCEL_S3_PATH
    if not excel_path:
        raise RuntimeError(
            f"No Excel config path set for {dag_id}. Set "
            f"{'REGRESSION_DAG1_EXCEL_S3_PATH' if dag_id == DAG1_ID else 'REGRESSION_DAG2_EXCEL_S3_PATH'}"
            f" to the s3a:// path of the regression Excel. Without it the DAG falls back to "
            f"its Param default (a placeholder like s3a://config-bucket/migration.xlsx) and "
            f"would migrate the wrong table set entirely.")

    if not looks_like_s3_uri(excel_path):
        raise RuntimeError(
            f"Excel path for {dag_id} is not an S3 URI: {excel_path!r}. Spark resolves a "
            f"schemeless path against its default filesystem and working directory, which "
            f"silently becomes something like "
            f"s3a://<bucket>/user/<you>/{excel_path} — the bucket appears twice and the "
            f"file is not there. Use {suggest_s3_uri(excel_path)!r} instead.")

    if GENERATE_EXCEL_CONFIGS and not globals().get("EXCEL_UPLOADED", False):
        raise RuntimeError(
            "Step 3b did not upload the Excel configs, so the path this run would point at "
            "does not exist. Re-run Step 3b and check what it refused to do.")

    if SPARK_AVAILABLE and not s3_exists(excel_path):
        raise RuntimeError(
            f"Excel config does not exist on S3: {excel_path}. parse_excel would fail with "
            f"PATH_NOT_FOUND. Re-run Step 3b, or point "
            f"{'REGRESSION_DAG1_EXCEL_S3_PATH' if dag_id == DAG1_ID else 'REGRESSION_DAG2_EXCEL_S3_PATH'}"
            f" at a workbook that is actually there.")

    if UPLOAD_ENV_FILE and ENV_CONFIG_S3_DIR and not looks_like_s3_uri(ENV_CONFIG_S3_DIR):
        raise RuntimeError(
            f"ENV_CONFIG_S3_DIR={ENV_CONFIG_S3_DIR!r} has no s3a:// scheme, so the env "
            f"file was written to a path the DAG does not read and {dag_id} would run on "
            f"the defaults in shared.py. Set REGRESSION_DAG_DEPLOY_S3_PREFIX="
            f"{suggest_s3_uri(DAG_DEPLOY_S3_PREFIX or ENV_CONFIG_S3_DIR)!r}")

    if UPLOAD_ENV_FILE and ENV_CONFIG_S3_DIR and SPARK_AVAILABLE:
        env_key = f"{ENV_CONFIG_S3_DIR.rstrip('/')}/env.{dag_id}"
        if not s3_exists(env_key):
            raise RuntimeError(
                f"No env file at {env_key}. {dag_id} would fall back to Airflow Variables "
                f"or the hardcoded defaults in shared.py — which is how "
                f"init_tracking_tables ends up on 'migration_tracking' instead of the "
                f"tracking database configured here. Run Step 2e.")

    expected_owner = AIRFLOW_VARIABLE_OVERRIDES.get("migration_dag_owner")
    if expected_owner:
        detail = airflow_request("GET", f"/dags/{dag_id}")
        owners = detail.get("owners") or []
        if isinstance(owners, str):
            owners = [owners]
        if expected_owner not in owners:
            raise RuntimeError(
                f"{dag_id} reports owners={owners}, not {expected_owner!r}. dag.owners is "
                f"what the pod_mutation_hook turns into HADOOP_USER_NAME/SPARK_USER, so the "
                f"tasks would run as {owners and owners[0] or 'data-migration'!r} and Ranger "
                f"would reject them.\n"
                f"  Fix, in order of preference:\n"
                f"    1. Redeploy with the owner baked in (per-deployment, no global state):\n"
                f"         python deploy.py --owner {expected_owner} "
                f"--env-file {ENV_FILE_NAME} --suffix <your-suffix>\n"
                f"    2. Set the Airflow Variable migration_dag_owner (Step 2d) — global, "
                f"affects everyone's runs.\n"
                f"  The env file from Step 2e CANNOT fix this: _resolve_dag_owner() reads "
                f"only Variable.get('migration_dag_owner'), never the MIGRATION_DAG_OWNER "
                f"env var. That env var sets config['owner'] and the kyuubi session user "
                f"(layer 1), not the pod identity Ranger checks (layer 2).\n"
                f"  Override this check with REGRESSION_SKIP_PREFLIGHT=true.")


def trigger_dag_run(dag_id, suffix, conf=None):
    assert_ready_to_trigger(dag_id)
    dag_run_id = f"regression_{RUN_TAG}_{suffix}"
    run_conf = dict(conf or {})
    if REGRESSION_DAG_OWNER:
        run_conf.setdefault("dag_owner", REGRESSION_DAG_OWNER)
    excel_path = DAG1_EXCEL_S3_PATH if dag_id == DAG1_ID else DAG2_EXCEL_S3_PATH
    if excel_path:
        run_conf.setdefault("excel_file_path", excel_path)
    payload = {"dag_run_id": dag_run_id, "conf": run_conf}
    if AIRFLOW_API_VERSION == "v2":
        payload["logical_date"] = None
    airflow_request("POST", f"/dags/{dag_id}/dagRuns", payload)
    RUN_IDS[suffix] = (dag_id, dag_run_id)
    print(f"Triggered {dag_id} as {dag_run_id}")
    if run_conf:
        print(f"  conf = {json.dumps(run_conf)}")
    return dag_run_id


def get_airflow_variable(key):
    try:
        return airflow_request("GET", f"/variables/{key}").get("value")
    except RuntimeError:
        return None


def set_airflow_variable(key, value):
    """Create or update an Airflow Variable.

    Global side effect: a Variable applies to every run of every DAG that reads it,
    not just this notebook's runs. Nothing calls this automatically.
    """
    try:
        airflow_request("PATCH", f"/variables/{key}", {"key": key, "value": value})
        print(f"Updated Airflow Variable {key} = {value!r}")
    except RuntimeError:
        airflow_request("POST", "/variables", {"key": key, "value": value})
        print(f"Created Airflow Variable {key} = {value!r}")
    return get_airflow_variable(key)


def wait_for_dag_reparse(dag_id, expected_owner):
    """Block until the scheduler re-parses the DAG and picks up the new owner.

    default_args['owner'] is evaluated at parse time, so a Variable change is invisible
    to a run triggered before the next parse.
    """
    deadline = time.monotonic() + DAG_REPARSE_TIMEOUT_SECONDS
    attempt = 0
    while time.monotonic() < deadline:
        attempt += 1
        detail = airflow_request("GET", f"/dags/{dag_id}")
        owners = detail.get("owners") or []
        if isinstance(owners, str):
            owners = [owners]
        if expected_owner in owners:
            print(f"  {dag_id}: reparsed after {attempt} check(s), owners={owners}")
            return True
        if attempt == 1 or attempt % 4 == 0:
            remaining = int(deadline - time.monotonic())
            print(f"  {dag_id}: owners={owners}, waiting for reparse "
                  f"({remaining}s left)")
        time.sleep(POLL_INTERVAL_SECONDS)
    print(f"  {dag_id}: TIMEOUT after {DAG_REPARSE_TIMEOUT_SECONDS}s — still not showing "
          f"{expected_owner!r}")
    return False


def apply_airflow_variable_overrides():
    """Push AIRFLOW_VARIABLE_OVERRIDES into Airflow, then wait for the DAG reparse.

    Writes shared Airflow state: a Variable applies to every run of every DAG that reads
    it, not only this notebook's runs. Previous values are captured in
    PREVIOUS_AIRFLOW_VARIABLES so restore_airflow_variables() can put them back.
    """
    if not AIRFLOW_VARIABLE_OVERRIDES:
        print("No Variable overrides configured — nothing to apply.")
        return True

    print("Writing shared Airflow Variables (affects every run of both DAGs):\n")
    owner_changed = False
    for key, value in AIRFLOW_VARIABLE_OVERRIDES.items():
        current = get_airflow_variable(key)
        PREVIOUS_AIRFLOW_VARIABLES.setdefault(key, current)
        if current == value:
            print(f"  {key}: already {value!r} — unchanged")
            continue
        print(f"  {key}: {current!r} -> {value!r}")
        set_airflow_variable(key, value)
        if key == "migration_dag_owner":
            owner_changed = True

    if not owner_changed:
        print("\nNo owner change, so no reparse needed.")
        return True

    owner = AIRFLOW_VARIABLE_OVERRIDES["migration_dag_owner"]
    print(f"\nWaiting for both DAGs to re-parse and adopt owner {owner!r}.")
    print("Until this completes, a triggered run still uses the old pod "
          "HADOOP_USER_NAME/SPARK_USER.")
    reparsed = all(wait_for_dag_reparse(dag_id, owner) for dag_id in ACTIVE_DAG_IDS)
    if not reparsed:
        print("\nAt least one DAG has not adopted the new owner, so triggering now would "
              "run as the old principal and fail Ranger the same way.")
        print("\n_resolve_dag_owner() reads the Variable at DAG PARSE time. If the parse "
              "context cannot read Variables, it silently falls through its except clause "
              "to the literal 'data-migration' and no wait will ever succeed. The tell is "
              "the DAG version never changing while the Variable is set.")
        print("\nThe deterministic alternative is to bake the owner into the deployed file "
              "instead, which needs no parse-time Variable lookup:")
        print(f"    python deploy.py --owner {AIRFLOW_VARIABLE_OVERRIDES.get('migration_dag_owner')}")
        print("deploy.py rewrites the `return 'data-migration'` literal in "
              "_resolve_dag_owner() before upload. Re-run this notebook afterwards.")
    return reparsed


def restore_airflow_variables():
    """Put back whatever apply_airflow_variable_overrides() replaced."""
    if not PREVIOUS_AIRFLOW_VARIABLES:
        print("Nothing captured to restore.")
        return
    for key, previous in PREVIOUS_AIRFLOW_VARIABLES.items():
        if previous is None:
            print(f"  {key}: was unset before this notebook — delete it manually in the "
                  f"Airflow UI to fully restore")
            continue
        set_airflow_variable(key, previous)


def resolve_s3_root_from_airflow():
    """Adopt the DAG's own migration_default_s3_bucket for the notebook's S3 paths.

    The notebook and the DAG must agree on the destination root: P-03/P-04 corrupt a
    Parquet file under the prefix DAG 1 wrote to, so a guess that differs from the DAG's
    value silently corrupts nothing and the case reports a false pass. Reading the
    Variable removes the guess.

    Only values that are not already valid S3 URIs are replaced, so an explicitly
    configured path is never clobbered.
    """
    global MIGRATION_S3_ROOT, EXCEL_OUTPUT_PREFIX
    global DAG1_EXCEL_S3_PATH, DAG2_EXCEL_S3_PATH, CORRUPT_EXCEL_S3_PATH
    global TRACKING_LOCATION, REPORT_LOCATION

    airflow_value = get_airflow_variable("migration_default_s3_bucket")
    print(f"Airflow migration_default_s3_bucket = {airflow_value!r}")
    print(f"notebook MIGRATION_S3_ROOT          = {MIGRATION_S3_ROOT!r}")

    if airflow_value and looks_like_s3_uri(airflow_value):
        if looks_like_s3_uri(MIGRATION_S3_ROOT):
            if MIGRATION_S3_ROOT.rstrip("/") != airflow_value.rstrip("/"):
                print("\n  WARNING: both are URIs but they differ. The notebook will keep its "
                      "own value, so P-03/P-04 will look for DAG 1's output under a prefix "
                      "the DAG did not write to. Make them match unless you know why they "
                      "differ.")
            else:
                print("\n  Match — notebook and DAG agree on the destination root.")
            return MIGRATION_S3_ROOT
        MIGRATION_S3_ROOT = airflow_value.rstrip("/")
        print(f"\n  Adopted the Airflow value: MIGRATION_S3_ROOT = {MIGRATION_S3_ROOT!r}")
    elif not looks_like_s3_uri(MIGRATION_S3_ROOT):
        print("\n  Airflow has no usable migration_default_s3_bucket either, and the "
              "notebook value is not a URI. Set MIGRATION_DEFAULT_S3_BUCKET to an "
              "s3a:// path — nothing that writes to S3 can run until then.")
        return MIGRATION_S3_ROOT
    else:
        print("\n  Airflow has no migration_default_s3_bucket; keeping the notebook value.")
        return MIGRATION_S3_ROOT

    if not looks_like_s3_uri(EXCEL_OUTPUT_PREFIX):
        EXCEL_OUTPUT_PREFIX = f"{MIGRATION_S3_ROOT}/regression_suite"
        print(f"  EXCEL_OUTPUT_PREFIX  = {EXCEL_OUTPUT_PREFIX!r}")
    if not looks_like_s3_uri(DAG1_EXCEL_S3_PATH):
        DAG1_EXCEL_S3_PATH = f"{EXCEL_OUTPUT_PREFIX}/regression_dag1.xlsx"
        print(f"  DAG1_EXCEL_S3_PATH   = {DAG1_EXCEL_S3_PATH!r}")
    if not looks_like_s3_uri(DAG2_EXCEL_S3_PATH):
        DAG2_EXCEL_S3_PATH = f"{EXCEL_OUTPUT_PREFIX}/regression_dag2.xlsx"
        print(f"  DAG2_EXCEL_S3_PATH   = {DAG2_EXCEL_S3_PATH!r}")
    if not looks_like_s3_uri(CORRUPT_EXCEL_S3_PATH):
        CORRUPT_EXCEL_S3_PATH = f"{EXCEL_OUTPUT_PREFIX}/corrupt_config.xlsx"
        print(f"  CORRUPT_EXCEL_S3_PATH= {CORRUPT_EXCEL_S3_PATH!r}")
    if not looks_like_s3_uri(TRACKING_LOCATION):
        TRACKING_LOCATION = f"{MIGRATION_S3_ROOT}/{TRACKING_DB}"
        print(f"  TRACKING_LOCATION    = {TRACKING_LOCATION!r}")
    if not looks_like_s3_uri(REPORT_LOCATION):
        REPORT_LOCATION = f"{MIGRATION_S3_ROOT}/migration_reports"
        print(f"  REPORT_LOCATION      = {REPORT_LOCATION!r}")
    return MIGRATION_S3_ROOT


def report_dag_owner_resolution():
    """Show which of the two owner layers is in play, and what each will resolve to."""
    variable_owner = get_airflow_variable("migration_dag_owner")
    tracking_variable = get_airflow_variable("migration_tracking_database")
    effective_tracking = tracking_variable or "migration_tracking"
    if effective_tracking != TRACKING_DB:
        print("TRACKING DATABASE MISMATCH")
        print(f"  this notebook reads      : {TRACKING_DB}")
        print(f"  the DAG will write to    : {effective_tracking}")
        print("  tracking_database takes no dag_run conf key, so the notebook value cannot")
        print("  reach the DAG. Either set TRACKING_DB to match, or run")
        print(f"    set_airflow_variable('migration_tracking_database', '{TRACKING_DB}')")
        print("  (global, and the owner needs CREATE privilege on the new database).")
        print()
    print("Owner layer 1 — per-run owner (config['owner'] / kyuubi session user)")
    print("  precedence: Variable/env migration_dag_owner > conf dag_owner > conf spark_user "
          "> 'data-migration'")
    print(f"  Variable migration_dag_owner = {variable_owner!r}")
    print(f"  conf dag_owner this notebook sends = {REGRESSION_DAG_OWNER or None!r}")
    if variable_owner:
        effective, why = variable_owner, "Airflow Variable wins over dag_run conf"
    elif REGRESSION_DAG_OWNER:
        effective, why = REGRESSION_DAG_OWNER, "from this notebook's conf dag_owner"
    else:
        effective, why = "data-migration", "nothing set — the fallback you are seeing"
    print(f"  -> effective per-run owner: {effective!r}  ({why})")
    print()
    print("Owner layer 2 — DAG-level owner: pod HADOOP_USER_NAME / SPARK_USER")
    print("  THIS is the principal Ranger authorizes and Keycloak maps groups for.")
    print("  Resolved at DAG parse time by _resolve_dag_owner(); reads only the Variable")
    print("  migration_dag_owner, never dag_run conf. A comma in the value is ignored.")
    print(f"  -> currently: {variable_owner or 'data-migration'!r}")
    if not variable_owner:
        print("  Ranger/Keycloak failures naming 'data-migration' are fixed HERE, not by")
        print("  REGRESSION_DAG_OWNER. Run:")
        print("    set_airflow_variable('migration_dag_owner', '<your-keycloak-user>')")
        print("  then wait for the next DAG parse. Global: affects every run of both DAGs.")


def wait_for_dag_run(dag_id, dag_run_id):
    deadline = time.monotonic() + DAG_RUN_TIMEOUT_SECONDS
    state = "unknown"
    while time.monotonic() < deadline:
        state = airflow_request("GET", f"/dags/{dag_id}/dagRuns/{dag_run_id}").get("state", "unknown")
        if state in TERMINAL_DAG_STATES:
            print(f"  {dag_id}/{dag_run_id} reached terminal state: {state}")
            return state
        print(f"  {dag_id}/{dag_run_id} state={state} — sleeping {POLL_INTERVAL_SECONDS}s")
        time.sleep(POLL_INTERVAL_SECONDS)
    raise TimeoutError(
        f"{dag_id}/{dag_run_id} still {state} after {DAG_RUN_TIMEOUT_SECONDS}s")


def run_dag_to_completion(dag_id, suffix, conf=None):
    dag_run_id = trigger_dag_run(dag_id, suffix, conf)
    return dag_run_id, wait_for_dag_run(dag_id, dag_run_id)


def task_instances(dag_id, dag_run_id):
    payload = airflow_request("GET", f"/dags/{dag_id}/dagRuns/{dag_run_id}/taskInstances")
    return payload.get("task_instances", [])


def task_instances_for(dag_id, dag_run_id, task_id):
    """Every instance of a task, including one per map_index for expanded tasks.

    The migration DAGs expand run_distcp_ssh, create_hive_tables and friends over the
    discovered tables, so a task_id resolves to many instances. Treating it as a single
    instance reads nothing: the unmapped log endpoint has no content for an expanded task.
    """
    return [ti for ti in task_instances(dag_id, dag_run_id)
            if ti.get("task_id") == task_id]


def task_instance(dag_id, dag_run_id, task_id):
    matches = task_instances_for(dag_id, dag_run_id, task_id)
    return matches[0] if matches else None


def max_try_number(dag_id, dag_run_id, task_id):
    """Highest attempt across all map indices — the retry count the case asserts on."""
    instances = task_instances_for(dag_id, dag_run_id, task_id)
    if not instances:
        return None
    return max(int(ti.get("try_number") or 0) for ti in instances)


def task_log(dag_id, dag_run_id, task_id, try_number=1, map_index=None):
    path = (f"/dags/{dag_id}/dagRuns/{dag_run_id}/taskInstances/"
            f"{task_id}/logs/{try_number}?full_content=true")
    if map_index is not None and map_index >= 0:
        path += f"&map_index={map_index}"
    try:
        return airflow_request("GET", path, as_text=True)
    except RuntimeError as err:
        suffix = "" if map_index in (None, -1) else f" map_index={map_index}"
        print(f"  log fetch failed for {task_id} try {try_number}{suffix}: {err}")
        return ""


def all_task_logs(dag_id, dag_run_id, task_id):
    """Concatenate every attempt of every mapped instance of a task."""
    instances = task_instances_for(dag_id, dag_run_id, task_id)
    if not instances:
        return ""
    chunks = []
    for instance in instances:
        map_index = instance.get("map_index")
        attempts = int(instance.get("try_number") or 1)
        for attempt in range(1, attempts + 1):
            chunks.append(task_log(dag_id, dag_run_id, task_id, attempt, map_index))
    print(f"  fetched {len(chunks)} log segment(s) for {task_id} "
          f"({len(instances)} instance(s))")
    return "\n".join(chunks)


SIZED_TABLE_RE = re.compile(
    r"\[DistCp\] Sized\s+(?P<table>[\w.]+):\s+(?P<mappers>\d+)\s+mappers\s+x\s+"
    r"(?P<bandwidth>\d+)\s+MB/s")
SIZED_PARTITION_RE = re.compile(
    r"\[DistCp\]\s+Partition\s+(?P<partition>\S+):\s+(?P<mappers>\d+)\s+mappers\s+x\s+"
    r"(?P<bandwidth>\d+)\s+MB/s")
EMPTY_SOURCE_RE = re.compile(r"\[DistCp\] EMPTY_SOURCE:\s+(?P<table>[\w.]+)")


def distcp_sizing_for_table(log_text, qualified_table):
    """(mappers, bandwidth) pairs from the table-level '[DistCp] Sized' lines.

    The DAG logs 'Sized <db>.<table>: N mappers x N MB/s', not the raw -m/-bandwidth
    flags, so the emitted distcp command line is not what this reads.
    """
    return [(int(m.group("mappers")), int(m.group("bandwidth")))
            for m in SIZED_TABLE_RE.finditer(log_text)
            if m.group("table").lower() == qualified_table.lower()]


def distcp_sizing_for_partition(log_text, partition_name):
    """(mappers, bandwidth) pairs from the per-partition lines.

    These are emitted with logger.debug, so they are absent unless the DAG's log level
    is DEBUG. An empty result therefore means 'not logged', not 'not sized'.
    """
    return [(int(m.group("mappers")), int(m.group("bandwidth")))
            for m in SIZED_PARTITION_RE.finditer(log_text)
            if m.group("partition") == partition_name]


def distcp_commands_for_path(log_text, path_fragment):
    """Emitted 'hadoop distcp' command lines that actually reference a path.

    Matching the bare substring 'distcp' is useless here: the fixture database is named
    distcp_sizing_db, so discovery and sizing lines mentioning any of its tables contain
    it. Only a real command line counts.
    """
    pattern = re.compile(r"hadoop\s+distcp[^\n]*" + re.escape(path_fragment))
    return [line for line in log_text.split("\n") if pattern.search(line)]


def empty_source_logged(log_text, qualified_table):
    return any(m.group("table").lower() == qualified_table.lower()
               for m in EMPTY_SOURCE_RE.finditer(log_text))

### Step 2b: Probe the Airflow API before triggering anything

Run this before any trigger cell. It pins the API base and auth mode, and fails loudly here —
with the exact status codes it saw — rather than at the first `POST`.

`detect_airflow_api` tries `/api/v2/version` then `/api/v1/version`. A `401`/`403` still
identifies the version (the path exists, the credentials just were not accepted yet); on a
`v2` `401` it attempts the Airflow 3 JWT exchange at `POST /auth/token` and switches to bearer
auth if that succeeds.

If no candidate answers, the probe automatically runs `explore_airflow_endpoints()`, which
prints what each candidate URL actually serves. The important distinction it draws is **JSON
from Airflow** versus an **HTML error page from a proxy/ingress** — an HTML 404 on every path,
including `/`, means the hostname is not routed to Airflow at all and no notebook setting can
fix it. Set `AIRFLOW_BASE_PATH` when Airflow sits under a path prefix (for example
`/airflow`), and give `AIRFLOW_API_BASE` the origin you use to open the Airflow UI.

The reachability assertion then does a real `GET /dags` so a credential or proxy problem
surfaces now. If this cell reports `405` on `/dags`, Airflow is behind a proxy that permits
only some methods and no trigger will work until that is changed.

In [ ]:
print("Probing the Airflow API...")
detect_airflow_api()

AIRFLOW_REACHABLE = False
if AIRFLOW_BASE_RESOLVED:
    print("\nReachability check: GET /dags")
    try:
        dags_payload = airflow_request("GET", "/dags?limit=1")
        print(f"  OK — Airflow reports {dags_payload.get('total_entries', '?')} DAG(s) "
              f"visible to this account")
        AIRFLOW_REACHABLE = True
    except Exception as err:
        print(f"  FAILED — {err}")

if AIRFLOW_REACHABLE:
    for dag_id in ACTIVE_DAG_IDS:
        try:
            detail = airflow_request("GET", f"/dags/{dag_id}")
            paused = detail.get("is_paused")
            print(f"  {dag_id}: present, is_paused={paused}"
                  + ("   <- unpause it or the run will sit queued forever" if paused else ""))
        except Exception as err:
            print(f"  {dag_id}: NOT REACHABLE — {err}")

print(f"\nAIRFLOW_REACHABLE = {AIRFLOW_REACHABLE}")
if not AIRFLOW_REACHABLE:
    print("Do not run the trigger cells until this probe passes — they will all raise.")

### Step 2c: Confirm which owner the runs will use

First resolves the S3 destination root, then prints both owner layers and which one wins,
so the `data-migration` fallback is visible before the DAGs are triggered rather than after.
Requires the Step 2b probe to have resolved the API base.

The S3 root is read from the Airflow Variable `migration_default_s3_bucket` — the same value
`get_config()` resolves into `default_s3_bucket`. Taking it from Airflow rather than
configuring it twice matters for P-03 and P-04: they corrupt a Parquet file under the prefix
DAG 1 wrote to, so a notebook-side guess that differs from the DAG's value would corrupt
nothing and the case would report a false pass. Values already set to a valid `s3a://` URI
are left alone; only unset or schemeless ones are filled in.

In [ ]:
if AIRFLOW_REACHABLE:
    resolve_s3_root_from_airflow()
    print()
    report_dag_owner_resolution()
else:
    print("Airflow not reachable — cannot read the migration_dag_owner Variable.")
    print(f"conf dag_owner this notebook would send = {REGRESSION_DAG_OWNER or None!r}")

### Step 2d: Apply the Airflow Variables from here

Two settings this notebook needs cannot travel in a dag_run conf, so they have to be Airflow
Variables:

| Variable | Set from | Why conf will not work |
|---|---|---|
| `migration_dag_owner` | `REGRESSION_DAG_OWNER` | Read at DAG **parse** time into `default_args['owner']` → `dag.owners` → the pod's `HADOOP_USER_NAME`/`SPARK_USER`. That is the principal Ranger authorizes, and it is fixed before the task runs. |
| `migration_tracking_database` | `TRACKING_DB` | `_var(...)` for `tracking_database` passes no `conf_key`, so it reads only the Variable or env. |

**This writes shared Airflow state.** A Variable applies to every run of every DAG that reads
it, not just this notebook's. The previous values are captured in
`PREVIOUS_AIRFLOW_VARIABLES`, and `restore_airflow_variables()` puts them back — though a
Variable that did not exist before can only be deleted from the Airflow UI.

Set `REGRESSION_APPLY_AIRFLOW_VARIABLES=false` to skip this — which is the default, because
Variables are cluster-global and affect other people's runs. **Step 2e generates a per-DAG env
file instead, which is scoped to your own deployment; prefer that.** Use this cell only when
redeploying is not an option and you accept the global side effect.

After changing the owner the cell **waits for both DAGs to re-parse**, polling
`GET /dags/{dag_id}` until `owners` reflects the new value. Skipping that wait is the
subtle failure mode: the Variable is set, the trigger fires immediately, and the pod is still
built from the old `dag.owners`, so Ranger rejects it exactly as before.

In [ ]:
VARIABLES_READY = False
if not APPLY_AIRFLOW_VARIABLES:
    print("REGRESSION_APPLY_AIRFLOW_VARIABLES=false — leaving Airflow Variables untouched.")
elif not AIRFLOW_REACHABLE:
    print("Airflow not reachable — cannot apply Variables. Fix Step 2b first.")
else:
    VARIABLES_READY = apply_airflow_variable_overrides()

print(f"\nVARIABLES_READY = {VARIABLES_READY}")
if APPLY_AIRFLOW_VARIABLES and AIRFLOW_REACHABLE and VARIABLES_READY:
    print("Re-checking owner resolution after the change:\n")
    report_dag_owner_resolution()

---
## Step 2e: Generate a per-DAG env file (isolated alternative to Airflow Variables)

Airflow Variables are cluster-global: `migration_tracking_database` set for this suite applies
to everyone's runs of every DAG that reads it. The repo already has an isolated mechanism, and
it is the better route here.

Each migration DAG loads, at import time:

```python
_dag_stem   = Path(__file__).stem
_config_dir = str(_dag_dir / "migrator_utils" / "migration_configs")
load_dotenv(os.path.join(_config_dir, "env.shared"))
load_dotenv(os.path.join(_config_dir, f"env.{_dag_stem}"), override=True)
```

So the scope is **per deployed DAG file**, keyed on its filename stem — not per DAG run, which
is not something the mechanism supports. Because `deploy.py` gives each person's deployment its
own suffixed filename (`es_source_to_s3_migration_tmp_sa_v1`), the matching
`env.es_source_to_s3_migration_tmp_sa_v1` is yours alone. Nobody else's runs see it. And
`override=True` means it beats `env.shared`.

### Precedence, so you know what actually wins

`_var()` resolves in this order — the env file sits **below** Airflow Variables:

```
dag_run conf (only for keys with a conf_key)  >  nx1_<key> Variables  >  <key> Variable
  >  env var (env.<dag_stem>, then env.shared)  >  hardcoded default
```

That has one consequence worth stating plainly: **if a global Variable already exists for a
key, it beats this env file.** If someone previously set `migration_tracking_database` as a
Variable — including an earlier run of this notebook's Step 2d — the env file is ignored for
that key until the Variable is deleted. Step 2c reports which layer is winning.

### Where the file goes

The DAG reads `<dag dir>/migrator_utils/migration_configs/env.<dag file stem>`, and the
deployed file is named after the DAG id, so the env file name is derived from the DAG id
rather than configured. Point `REGRESSION_DAG_DEPLOY_S3_PREFIX` at the prefix your DAGs are
deployed under and the notebook appends `/migrator_utils/migration_configs`:

```
REGRESSION_DAG_DEPLOY_S3_PREFIX=s3a://<bucket>/airflow/dags
  -> s3a://<bucket>/airflow/dags/migrator_utils/migration_configs/env.<dag_id>
```

Set `REGRESSION_ENV_CONFIG_S3_DIR` instead if the config directory is not under the DAG
prefix. One file is written per DAG in `ACTIVE_DAG_IDS`, with the same content — both DAGs
read the same keys.

`REGRESSION_UPLOAD_ENV_FILE=false` skips the upload and leaves the `deploy.py --env-file`
route below.

### Or let `deploy.py` publish it

The env file has to sit next to the deployed DAG on the worker's local disk, so the notebook
cannot place it directly. `deploy.py --env-file` is the supported publisher: it uploads your
local file to
`{dags_prefix}{package}/migration_configs/env.{dag_stem}_{suffix}`, which is exactly the name
the deployed DAG looks for.

This cell writes the file locally when `REGRESSION_ENV_FILE_DIR` points at your checkout, and
always prints the contents so you can copy it across if the notebook is not running on the
machine you deploy from.

Only non-empty values are written. That is deliberate: `override=True` means writing
`MIGRATION_DEFAULT_S3_BUCKET=` with an empty value would blank out whatever `env.shared`
provides, which is worse than not writing the key at all.

### What this file cannot do: the Ranger principal

`MIGRATION_DAG_OWNER` in this file reaches **layer 1 only** — `config['owner']` and
`spark.sql.kyuubi.session.user`, via `_var('migration_dag_owner', 'MIGRATION_DAG_OWNER', ...)`
at `shared.py:379`.

It does **not** set `dag.owners`, which is what the `pod_mutation_hook` turns into the pod's
`HADOOP_USER_NAME`/`SPARK_USER` — the identity Ranger authorizes and Keycloak resolves groups
for. That comes from `_resolve_dag_owner()`, which reads
`Variable.get('migration_dag_owner')` at parse time and never consults the environment.

So a Ranger `AccessControlException` naming `data-migration` is not fixable from this file.
Two routes, and the first preserves the isolation this file exists for:

```bash
python deploy.py --owner <user> --env-file env.regression_suite --suffix <your-suffix>
```

`--owner` rewrites the `return 'data-migration'` literal inside `_resolve_dag_owner()` before
upload, so it is baked into your deployment with no Airflow Variable and no effect on anyone
else. The alternative is the global Variable via Step 2d.

Note the name mapping — the notebook's DistCp knobs use the same names as
`setup-distcp-test-data.sh`, but the DAG reads `MIGRATION_DISTCP_*`, so they are translated
here.

### The three S3 locations that must all be set

`shared.py` has exactly three `_var` defaults pointing at the placeholder bucket
`s3a://data-lake`, and every one of them has to be overridden or the DAG writes to a bucket
you do not own:

| Key | Default | Used by |
|---|---|---|
| `MIGRATION_DEFAULT_S3_BUCKET` | `s3a://data-lake` | the migration destination |
| `MIGRATION_TRACKING_LOCATION` | `s3a://data-lake/migration_tracking` | the Iceberg tracking tables |
| `MIGRATION_REPORT_LOCATION` | `s3a://data-lake/migration_reports` | `generate_html_report` |

Missing the third is easy to do, because the DAG gets all the way through migration and
validation before `generate_html_report` fails — and it fails as
`AWSRedirectException: Received permanent redirect response to region [...]`, which reads like
an S3 region misconfiguration rather than a wrong bucket. All three are written by this cell,
derived from `MIGRATION_S3_ROOT`, and the config cell warns if any is empty.

In [ ]:
import pathlib

ENV_FILE_ENTRIES = [
    ("MIGRATION_TRACKING_DATABASE", TRACKING_DB),
    ("MIGRATION_TRACKING_LOCATION", TRACKING_LOCATION),
    ("MIGRATION_REPORT_LOCATION", REPORT_LOCATION),
    ("MIGRATION_EMAIL_RECIPIENTS", EMAIL_RECIPIENTS),
    ("MIGRATION_DAG_OWNER", REGRESSION_DAG_OWNER),
    ("MIGRATION_DEFAULT_S3_BUCKET", MIGRATION_S3_ROOT),
    ("MIGRATION_INCLUDE_DB_IN_PATH", "true" if MIGRATION_INCLUDE_DB_IN_PATH else "false"),
    ("CLUSTER_SSH_CONN_ID", CLUSTER_SSH_CONN_ID),
    ("MIGRATION_DISTCP_TARGET_BYTES_PER_MAPPER", str(DISTCP_TARGET_BYTES_PER_MAPPER)),
    ("MIGRATION_DISTCP_MIN_MAPPERS", str(DISTCP_MIN_MAPPERS)),
    ("MIGRATION_DISTCP_MAX_MAPPERS", str(DISTCP_MAX_MAPPERS)),
    ("MIGRATION_DISTCP_TARGET_AGGREGATE_MBPS", str(DISTCP_TARGET_AGGREGATE_MBPS)),
]


GENERATED_ENV_MARKER = "Generated by notebooks/regression_test_suite.ipynb"

MUST_BE_S3_URI = {
    "MIGRATION_DEFAULT_S3_BUCKET",
    "MIGRATION_TRACKING_LOCATION",
    "MIGRATION_REPORT_LOCATION",
}

VARIABLE_FOR_ENV_KEY = {
    "MIGRATION_TRACKING_DATABASE": "migration_tracking_database",
    "MIGRATION_TRACKING_LOCATION": "migration_tracking_location",
    "MIGRATION_REPORT_LOCATION": "migration_report_location",
    "MIGRATION_EMAIL_RECIPIENTS": "migration_email_recipients",
    "MIGRATION_DAG_OWNER": "migration_dag_owner",
    "MIGRATION_DEFAULT_S3_BUCKET": "migration_default_s3_bucket",
    "MIGRATION_INCLUDE_DB_IN_PATH": "migration_include_db_in_path",
    "CLUSTER_SSH_CONN_ID": "cluster_ssh_conn_id",
    "MIGRATION_DISTCP_TARGET_BYTES_PER_MAPPER": "migration_distcp_target_bytes_per_mapper",
    "MIGRATION_DISTCP_MIN_MAPPERS": "migration_distcp_min_mappers",
    "MIGRATION_DISTCP_MAX_MAPPERS": "migration_distcp_max_mappers",
    "MIGRATION_DISTCP_TARGET_AGGREGATE_MBPS": "migration_distcp_target_aggregate_mbps",
}


def audit_env_file_precedence():
    """For each key the env file sets, report whether an Airflow Variable outranks it.

    _var() puts Variables above env files, so a leftover Variable silently wins and the
    env file appears to have no effect. This is the check for 'I set it and nothing
    changed'.
    """
    print("Key".ljust(42) + "env file".ljust(34) + "Airflow Variable -> winner")
    print("-" * 118)
    shadowed = []
    schemeless = []
    for key, value in ENV_FILE_ENTRIES:
        variable_name = VARIABLE_FOR_ENV_KEY.get(key)
        variable_value = get_airflow_variable(variable_name) if variable_name else None
        env_shown = (str(value)[:30] + "..") if len(str(value)) > 32 else str(value or "-")
        if variable_value is not None:
            shadowed.append((key, variable_name, variable_value))
            winning_value = variable_value
            verdict = f"{str(variable_value)[:28]!r} -> VARIABLE WINS"
        else:
            winning_value = value
            verdict = "(unset) -> env file wins"
        if key in MUST_BE_S3_URI and winning_value and not looks_like_s3_uri(winning_value):
            schemeless.append((key, winning_value, variable_name if variable_value is not None
                               else "env file"))
            verdict += "  [NOT AN s3a:// URI]"
        print(f"{key.ljust(42)}{env_shown.ljust(34)}{verdict}")

    if schemeless:
        print()
        print(f"{len(schemeless)} winning value(s) are not S3 URIs. Spark resolves a "
              f"schemeless path against its default filesystem and working directory, so "
              f"the data lands under s3a://<bucket>/user/<you>/<the path>, with the bucket "
              f"name repeated — the write succeeds and the reported path is unreachable:")
        for key, winning_value, source in schemeless:
            print(f"  {key} = {winning_value!r}  (from {source})")
            print(f"    should be: {suggest_s3_uri(winning_value)!r}")
    if shadowed:
        print(f"\n{len(shadowed)} key(s) are shadowed by Airflow Variables. The env file "
              f"cannot take effect for these until the Variables are deleted:")
        for key, variable_name, variable_value in shadowed:
            print(f"  Variable {variable_name!r} = {variable_value!r}  (shadows {key})")
        print("\nDelete them in the Airflow UI under Admin -> Variables, or run "
              "restore_airflow_variables() if this notebook created them.")
    else:
        print("\nNo Variables shadow the env file — every key above resolves from it, "
              "provided the file was deployed.")
    return shadowed


def build_env_file_text():
    """Render the env file, omitting empty values so env.shared is not blanked out."""
    header = [
        "# Generated by notebooks/regression_test_suite.ipynb — do not edit by hand.",
        f"# Run tag {RUN_TAG}.",
        "#",
        "# Publish with deploy.py, which renames it to env.<dag_stem>_<suffix> so the",
        "# deployed DAG picks it up via load_dotenv(..., override=True):",
        f"#   python deploy.py --env-file {ENV_FILE_NAME} --suffix <your-suffix> \\",
        f"#       --owner {REGRESSION_DAG_OWNER or '<your-keycloak-user>'}",
        "#",
        "# Scope is the deployed DAG filename, so these values affect only your",
        "# deployment. An existing Airflow Variable for the same key still outranks",
        "# this file — see Step 2c.",
        "#",
        "# MIGRATION_DAG_OWNER here sets config['owner'] and the kyuubi session user",
        "# only. It does NOT set dag.owners / HADOOP_USER_NAME, which is what Ranger",
        "# authorizes: _resolve_dag_owner() reads Variable.get('migration_dag_owner')",
        "# at parse time and never this env var. Use deploy.py --owner for that.",
        "",
    ]
    body, omitted = [], []
    for key, value in ENV_FILE_ENTRIES:
        if str(value or "").strip():
            body.append(f"{key}={value}")
        else:
            omitted.append(key)
    return "\n".join(header + body) + "\n", omitted


def env_file_targets():
    """(dag_id, s3 key) for every DAG that will run.

    load_dotenv reads env.<Path(__file__).stem>, and deploy.py names the deployed file
    after the DAG id, so the env file name is derived from the DAG id rather than
    configured separately. Both DAGs read the same keys (tracking database, S3 root,
    owner), so both get the same content.
    """
    if not ENV_CONFIG_S3_DIR:
        return []
    return [(dag_id, f"{ENV_CONFIG_S3_DIR.rstrip('/')}/env.{dag_id}")
            for dag_id in ACTIVE_DAG_IDS]


env_file_text, env_file_omitted = build_env_file_text()

ENV_FILE_WRITTEN_TO = None
if ENV_FILE_DIR:
    target_dir = pathlib.Path(ENV_FILE_DIR).expanduser()
    if target_dir.is_dir():
        target = target_dir / ENV_FILE_NAME
        target.write_text(env_file_text, encoding="utf-8")
        ENV_FILE_WRITTEN_TO = str(target)
        print(f"Wrote {ENV_FILE_WRITTEN_TO}\n")
    else:
        print(f"REGRESSION_ENV_FILE_DIR={ENV_FILE_DIR!r} is not a directory — "
              f"printing the contents only.\n")
else:
    print("REGRESSION_ENV_FILE_DIR is unset, so nothing was written to disk. Set it to your "
          "nx1-data-migrator checkout to have the file written there, or copy the text "
          "below.\n")

if env_file_omitted:
    print(f"Omitted (empty, so env.shared keeps its value): {', '.join(env_file_omitted)}\n")

print("----- env.<dag_id> -----")
print(env_file_text, end="")
print("----- end -----")

ENV_FILES_UPLOADED = {}
if not UPLOAD_ENV_FILE:
    print()
    print("REGRESSION_UPLOAD_ENV_FILE=false — not uploading. Publish with "
          "deploy.py --env-file <local file> instead.")
elif not ENV_CONFIG_S3_DIR:
    print()
    print("No env config directory configured, so nothing was uploaded. Set "
          "REGRESSION_DAG_DEPLOY_S3_PREFIX to the prefix your DAGs are deployed under "
          "(the notebook appends /migrator_utils/migration_configs), or set "
          "REGRESSION_ENV_CONFIG_S3_DIR to that directory directly.")
elif not looks_like_s3_uri(ENV_CONFIG_S3_DIR):
    print()
    print(f"Refusing to upload: ENV_CONFIG_S3_DIR={ENV_CONFIG_S3_DIR!r} has no s3a:// "
          f"scheme.")
    print("  Writing there would succeed but land under "
          "s3a://<bucket>/user/<you>/<the path>, which the DAG never reads. The DAG would "
          "then fall back to the defaults in shared.py — tracking database "
          "'migration_tracking' and report location 's3a://data-lake/migration_reports', "
          "the latter failing with a 301 redirect.")
    print(f"  Set REGRESSION_DAG_DEPLOY_S3_PREFIX="
          f"{suggest_s3_uri(DAG_DEPLOY_S3_PREFIX or ENV_CONFIG_S3_DIR)!r}")
else:
    print()
    try:
        for dag_id, s3_key in env_file_targets():
            existing = s3_read_text(s3_key) if s3_exists(s3_key) else None
            if (existing is not None and GENERATED_ENV_MARKER not in existing
                    and not OVERWRITE_FOREIGN_ENV):
                print(f"  {dag_id}: REFUSED — {s3_key} already exists and was not written "
                      f"by this notebook.")
                print(f"    Overwriting it would replace someone else's configuration for "
                      f"{dag_id}. This matters most for an unsuffixed, shared DAG id: "
                      f"writing env.{dag_id} there changes that DAG for everyone.")
                print("    Point REGRESSION_DAG2_ID (or DAG1) at your own suffixed "
                      "deployment, or set REGRESSION_OVERWRITE_FOREIGN_ENV=true if you "
                      "really mean to replace it.")
                continue
            s3_write_bytes(s3_key, env_file_text.encode("utf-8"))
            ENV_FILES_UPLOADED[dag_id] = s3_key
            action = "overwrote" if existing is not None else "created"
            print(f"  {dag_id} -> {s3_key}  ({action})")
        for dag_id, s3_key in list(ENV_FILES_UPLOADED.items()):
            if not s3_exists(s3_key):
                print(f"  WARNING: wrote {s3_key} but it does not read back — the DAG "
                      f"will not see it.")
                ENV_FILES_UPLOADED.pop(dag_id)
        if ENV_FILES_UPLOADED:
            print()
            print("Uploaded and verified readable. Two caveats before this takes effect:")
            print("  1. The DAG bundle has to re-sync from S3 and the DAG has to be "
                  "re-parsed — load_dotenv runs at module import, not per run.")
            print("  2. An Airflow Variable for the same key still outranks the env "
                  "file. The audit below lists any that do.")
    except SparkUnavailable as err:
        print(f"Cannot upload without Spark ({err}); nothing was written.")

if APPLY_AIRFLOW_VARIABLES:
    print("\nNOTE: REGRESSION_APPLY_AIRFLOW_VARIABLES is true, so Step 2d also wrote "
          "cluster-global Airflow Variables, and those outrank this env file. Set it to "
          "false to rely on the env file alone.")

if globals().get("AIRFLOW_REACHABLE"):
    print()
    audit_env_file_precedence()
else:
    print("\nAirflow not reachable, so the Variable-shadowing audit was skipped. Run "
          "Step 2b first — a leftover Variable silently outranks this whole file.")

### After generating: the file has to be deployed, and re-read

Two things have to happen before a DAG run sees any of this, and both are easy to skip:

1. **Publish it.** `deploy.py --env-file <name>` uploads it to
   `{dags_prefix}{package}/migration_configs/env.{dag_stem}_{suffix}`. Writing the file next
   to the notebook changes nothing by itself.
2. **Let the DAG re-import.** `load_dotenv` runs at module import, so the values are picked
   up on the next DAG parse, not on the next run of an already-parsed DAG.

If a value still looks wrong after both, the order to check is:

| Symptom | Check |
|---|---|
| The value is the DAG's hardcoded default | The env file was not deployed, or the DAG has not re-parsed |
| The value is something else entirely | An Airflow Variable shadows it — the audit above lists them |

`generate_html_report` is the task that exposes a missed `MIGRATION_REPORT_LOCATION`, and it
does so confusingly: the default is `s3a://data-lake/migration_reports`, a bucket you do not
own, and S3 answers a request for someone else's bucket with a **301 redirect naming that
bucket's region**. The result is `AWSRedirectException: Received permanent redirect response
to region [...]`, which reads like a region misconfiguration rather than a wrong bucket name.
The DAG does not log the report path before writing, so the exception alone cannot tell you
which bucket it tried.

---
## Step 3: Expected-results table

One list of dicts, transcribed from the reference tables in the setup scripts. Nothing
downstream re-derives an expectation — every check in Step 6 reads this table.

Field meanings:

| Field | Meaning |
|---|---|
| `suite` / `sid` | Scenario id exactly as the source script prints it. The two `[6/11]` reference tables both number from `TC-1`, so `suite` disambiguates them. |
| `source` | `db.table` the check queries, or `None` for run-level checks. |
| `phase` | `DAG1` → `migration_table_status`; `DAG2` → `iceberg_migration_table_status`; `FAULT` → the fault-injection re-run of DAG 1; `UNIT` / `MANUAL` → no DAG run. |
| `expected_status` | Expected `overall_status`. |
| `expected_rows` | Exact source row count, or the string `">0"` where `[5/11]` only asserts non-empty. |
| `expected_parts` | Source-side `SHOW PARTITIONS` count from `[5/11]` / `[10/11]`. |
| `expected_mappers` / `expected_bandwidth` | From `size_job()` over the on-disk bytes/files `setup-distcp-test-data.sh` `[3/5]` loads. |
| `checks` | Which verifications apply. |

`">0"` is used, rather than a number, wherever the script itself only distinguishes empty from
non-empty (the `empty_tables` set in `[5/11]`); the `[10/11]` `checks` list does give exact
counts and those are carried through verbatim.

Sizing expectations are computed by the `size_distcp_job` port rather than pasted as literals,
so changing a knob in Step 0 moves the expectation with it — exactly what the bash `[5/5]`
block does.

In [ ]:
def scenario(suite, sid, source=None, phase="DAG1", expected_status=None, expected_rows=None,
             expected_parts=None, partition_filter=None, dest_database=None,
             expected_mappers=None,
             expected_bandwidth=None, checks=("status",), note="", enabled=True,
             blocked_reason=None, struct_column=None, expected_columns=None,
             expected_partition_names=None, expected_attempts=None, log_task=None,
             log_must_contain=None, log_must_not_contain=None, log_any_of=None,
             source_bytes=None, source_files=None):
    if source_bytes is not None:
        expected_mappers, expected_bandwidth = size_distcp_job(source_bytes, source_files or 0)
    return {
        "suite": suite, "sid": sid, "source": source, "phase": phase,
        "dest_database": dest_database,
        "expected_status": expected_status, "expected_rows": expected_rows,
        "expected_parts": expected_parts, "partition_filter": partition_filter,
        "expected_mappers": expected_mappers, "expected_bandwidth": expected_bandwidth,
        "checks": tuple(checks), "note": note, "enabled": enabled,
        "blocked_reason": blocked_reason, "struct_column": struct_column,
        "expected_columns": expected_columns,
        "expected_partition_names": expected_partition_names,
        "expected_attempts": expected_attempts, "log_task": log_task,
        "log_must_contain": log_must_contain, "log_must_not_contain": log_must_not_contain,
        "log_any_of": log_any_of, "source_bytes": source_bytes, "source_files": source_files,
    }


TC33_COLUMNS = ["emp_id", "name", "dept", "job_title", "salary",
                "hire_date", "is_active", "cost_centre"]

EMPTY_PARTITIONED_PARTS = {
    "region=EU/year=2024/month=1",
    "region=US/year=2024/month=1",
    "region=US/year=2024/month=2",
}

SCENARIOS = [
    scenario("DAG-SCENARIO", "TC-1a", "sales_db.orders", dest_database="sales_db_dest",
             expected_status="VALIDATED",
             expected_rows=">0", checks=("status", "rows"),
             note="Wildcard sales_db/* — full DB migration"),
    scenario("DAG-SCENARIO", "TC-1b", "sales_db.orders_empty", expected_status="EMPTY_SOURCE",
             expected_rows=0, checks=("status", "rows"),
             note="Wildcard sales_db/* — empty member of the wildcard set"),
    scenario("DAG-SCENARIO", "TC-2", "hr_db.employees", dest_database="hr_db_dest",
             expected_status="VALIDATED",
             expected_rows=">0", checks=("status", "rows"),
             note="Single explicit table, non-partitioned"),
    scenario("DAG-SCENARIO", "TC-3a", "hr_db.employees", dest_database="hr_db_dest",
             expected_status="VALIDATED",
             expected_rows=">0", checks=("status", "rows"),
             note="Comma-separated tables — populated member"),
    scenario("DAG-SCENARIO", "TC-3b", "hr_db.departments", expected_status="EMPTY_SOURCE",
             expected_rows=0, checks=("status", "rows"),
             note="Comma-separated tables — EMPTY_SOURCE member"),
    scenario("DAG-SCENARIO", "TC-4", "sales_db.orders", dest_database="sales_db_dest",
             expected_status="VALIDATED",
             expected_rows=">0", partition_filter="year=2024 AND month=1",
             checks=("status", "rows"), note="Partition filter active"),
    scenario("DAG-SCENARIO", "TC-5", "sales_db.orders", dest_database="sales_db_dest",
             expected_status="SKIPPED",
             partition_filter="year=2099", checks=("status",),
             note="Partition filter matches 0 partitions -> SKIPPED"),
    scenario("DAG-SCENARIO", "TC-6", "analytics_db.sessions", expected_status="VALIDATED",
             expected_rows=0, expected_parts=0, checks=("status", "rows", "parts"),
             note="Unregistered source partitions -> DAG runs MSCK REPAIR. The source-side "
                  "COUNT(*) is 0 by design: the data files exist on HDFS but no partition "
                  "is registered, so Hive sees nothing until the DAG repairs the "
                  "destination. 0 rows here is the fixture working, not a failure; the "
                  "migration itself is asserted by overall_status=VALIDATED."),
    scenario("DAG-SCENARIO", "TC-7", "analytics_db.events", expected_status="VALIDATED",
             expected_rows=">0", checks=("status", "rows"), note="3-level partition key"),
    scenario("DAG-SCENARIO", "TC-8", "analytics_db.events", expected_status="VALIDATED",
             expected_rows=">0", partition_filter="region='US' AND year=2024",
             checks=("status", "rows"), note="Filter on a 3-level partition key"),
    scenario("DAG-SCENARIO", "TC-9", "logs_db.error_logs_empty", expected_status="EMPTY_SOURCE",
             expected_rows=0, checks=("status", "rows"),
             note="EMPTY_SOURCE: dt-partitioned, 0 data files"),
    scenario("DAG-SCENARIO", "TC-10", "logs_db.metrics_empty_nonpartitioned",
             expected_status="EMPTY_SOURCE", expected_rows=0, checks=("status", "rows"),
             note="EMPTY_SOURCE: flat non-partitioned"),
    scenario("DAG-SCENARIO", "TC-11", "analytics_db.events_empty_partitioned",
             expected_status="EMPTY_SOURCE", expected_rows=0, expected_parts=3,
             expected_partition_names=EMPTY_PARTITIONED_PARTS,
             checks=("status", "rows", "parts", "partition_names"),
             note="EMPTY_SOURCE: 3 partitions registered, all empty (no data files)"),
    scenario("DAG-SCENARIO", "TC-11b", "analytics_db.events_parquet_empty_partitioned",
             expected_status="EMPTY_SOURCE", expected_rows=0, expected_parts=3,
             expected_partition_names=EMPTY_PARTITIONED_PARTS,
             checks=("status", "rows", "parts", "partition_names"),
             note="Same as TC-11 but STORED AS PARQUET (called TC-34 in the [5/11] block)"),
    scenario("DAG-SCENARIO", "TC-12", "sales_db.orders_empty", expected_status="EMPTY_SOURCE",
             expected_rows=0, checks=("status", "rows"),
             note="EMPTY_SOURCE: year/month partitioned"),
    scenario("DAG-SCENARIO", "TC-13", "analytics_db.events_orc", expected_status="VALIDATED",
             expected_rows=">0", checks=("status", "rows"), note="ORC format"),
    scenario("DAG-SCENARIO", "TC-14", "hr_db.employees_parquet", expected_status="VALIDATED",
             expected_rows=">0", checks=("status", "rows"),
             note="PARQUET, schema inferred from files"),
    scenario("DAG-SCENARIO", "TC-15", "hr_db.employees_avro", expected_status="VALIDATED",
             expected_rows=">0", checks=("status", "rows"), note="AVRO format"),
    scenario("DAG-SCENARIO", "TC-16", "logs_db.app_logs", expected_status="VALIDATED",
             expected_rows=">0", checks=("status", "rows"),
             note="TEXTFILE with serde field.delim"),
    scenario("DAG-SCENARIO", "TC-17", "logs_db.app_logs", expected_status="VALIDATED",
             expected_rows=">0", partition_filter="dt>='2024-01-15'",
             checks=("status", "rows"), note="String dt partition filter"),
    scenario("DAG-SCENARIO", "TC-18a", "sales_db.orders", dest_database="sales_db_dest",
             expected_status="VALIDATED",
             expected_rows=">0", checks=("status", "rows"), note="Glob pattern orders*"),
    scenario("DAG-SCENARIO", "TC-18b", "sales_db.orders_empty", expected_status="EMPTY_SOURCE",
             expected_rows=0, checks=("status", "rows"),
             note="Glob pattern orders* — empty match"),
    scenario("DAG-SCENARIO", "TC-19", "hr_db.employees", dest_database="hr_renamed",
             expected_status="VALIDATED",
             expected_rows=">0", checks=("status", "rows"),
             note="Different dest_db (hr_renamed) and alternate bucket"),
    scenario("DAG-SCENARIO", "TC-20", "sales_db.orders", dest_database="sales_db_dest",
             expected_status="VALIDATED",
             expected_rows=">0", checks=("status", "rows"), note="Custom S3 endpoint"),
    scenario("DAG-SCENARIO", "TC-21a", "logs_db.app_logs", expected_status="VALIDATED",
             expected_rows=">0", partition_filter="dt='2024-01-01'",
             checks=("status", "rows"), note="Wildcard logs_db/* plus partition_filter"),
    scenario("DAG-SCENARIO", "TC-21b", "logs_db.error_logs_empty", expected_status="EMPTY_SOURCE",
             expected_rows=0, partition_filter="dt='2024-01-01'", checks=("status", "rows"),
             note="Wildcard plus filter — empty dt-partitioned member"),
    scenario("DAG-SCENARIO", "TC-21c", "logs_db.metrics_empty_nonpartitioned",
             expected_status="EMPTY_SOURCE", expected_rows=0, checks=("status", "rows"),
             note="Wildcard plus filter — empty non-partitioned member"),
    scenario("DAG-SCENARIO", "TC-22a", "analytics_db.sessions", expected_status="VALIDATED",
             expected_rows=0, partition_filter="year=2024 AND month=1",
             checks=("status", "rows"),
             note="CSV plus filter, comma-separated table list. Source COUNT(*) is 0 for "
                  "the same unregistered-partition reason as TC-6."),
    scenario("DAG-SCENARIO", "TC-22b", "analytics_db.events", expected_status="VALIDATED",
             expected_rows=">0", partition_filter="year=2024 AND month=1",
             checks=("status", "rows"), note="CSV plus filter, comma-separated table list"),
    scenario("DAG-SCENARIO", "TC-23", "sales_db.orders", dest_database="sales_db_dest",
             expected_status="VALIDATED",
             expected_rows=">0", checks=("status", "rows"),
             note="Incremental re-run (2nd run over the same table)"),
    scenario("DAG-SCENARIO", "TC-24", "hr_db.employees", dest_database="hr_db_dest",
             expected_status="VALIDATED",
             expected_rows=">0", checks=("status", "rows"),
             note="\\N NULL values survive the round trip"),
    scenario("DAG-SCENARIO", "TC-25", "sales_db.orders", dest_database="sales_db",
             expected_status="VALIDATED",
             expected_rows=">0", checks=("status", "rows"),
             note="Blank dest_db defaults to the source database"),

    scenario("HIVE-TYPE-DDL", "TC-6", "struct_db.simple_struct", expected_status="VALIDATED",
             expected_rows=">0", struct_column="info",
             checks=("status", "rows", "struct_ddl"),
             note="struct<a:int,b:string> -> colon removal"),
    scenario("HIVE-TYPE-DDL", "TC-7", "struct_db.simple_struct", expected_status="VALIDATED",
             expected_rows=">0", struct_column="info",
             checks=("status", "rows", "struct_ddl"),
             note="struct<a:int,b:string> -> all fields converted"),
    scenario("HIVE-TYPE-DDL", "TC-9", "struct_db.array_of_struct", expected_status="VALIDATED",
             expected_rows=">0", struct_column="items",
             checks=("status", "rows", "struct_ddl"), note="array<struct<x:int,y:string>>"),
    scenario("HIVE-TYPE-DDL", "TC-11", "struct_db.map_of_struct", expected_status="VALIDATED",
             expected_rows=">0", struct_column="tags",
             checks=("status", "rows", "struct_ddl"), note="map<string,struct<a:int>>"),
    scenario("HIVE-TYPE-DDL", "TC-12", "struct_db.deep_nested_struct",
             expected_status="VALIDATED", expected_rows=">0", struct_column="payload",
             checks=("status", "rows", "struct_ddl"),
             note="struct<a:array<struct<b:map<string,int>>>>"),
    scenario("HIVE-TYPE-DDL", "TC-13", "struct_db.array_of_array_struct",
             expected_status="VALIDATED", expected_rows=">0", checks=("status", "rows"),
             note="array<array<struct<x:int>>>"),
    scenario("HIVE-TYPE-DDL", "TC-14", "struct_db.struct_with_decimal",
             expected_status="VALIDATED", expected_rows=">0", struct_column="price_info",
             checks=("status", "rows", "struct_ddl"),
             note="decimal(18,4) comma must not split struct fields"),
    scenario("HIVE-TYPE-DDL", "TC-15", "struct_db.struct_with_varchar",
             expected_status="VALIDATED", expected_rows=">0", checks=("status", "rows"),
             note="varchar(255)/char(10) pass through unchanged"),
    scenario("HIVE-TYPE-DDL", "TC-16", "struct_db.already_converted",
             expected_status="VALIDATED", expected_rows=">0", struct_column="info",
             checks=("status", "rows", "struct_ddl"),
             note="Idempotent: already space-separated DDL left unchanged"),
    scenario("HIVE-TYPE-DDL", "TC-20", "struct_db.mixed_case_struct",
             expected_status="VALIDATED", expected_rows=">0", checks=("status", "rows"),
             note="STRUCT<A:INT> uppercase recognised"),
    scenario("HIVE-TYPE-DDL", "TC-26", "struct_db.flex_rules_result_mini",
             expected_status="VALIDATED", expected_rows=">0",
             struct_column="customerAttributes", checks=("status", "rows", "struct_ddl"),
             note="customerAttributes real type string parses"),
    scenario("HIVE-TYPE-DDL", "TC-27", "struct_db.flex_rules_result_mini",
             expected_status="VALIDATED", expected_rows=">0", struct_column="response",
             checks=("status", "rows", "struct_ddl"),
             note="response nested array-of-struct parses"),
    scenario("HIVE-TYPE-DDL", "TC-28a", "struct_db.flex_rules_result_full",
             expected_status="VALIDATED", expected_rows=">0",
             struct_column="customerAttributes", checks=("status", "rows", "struct_ddl"),
             note="Full CREATE EXTERNAL TABLE succeeds; DESCRIBE matches"),
    scenario("HIVE-TYPE-DDL", "TC-28b", "struct_db.flex_rules_result_full",
             expected_status="VALIDATED", expected_rows=">0", struct_column="response",
             checks=("status", "rows", "struct_ddl"),
             note="Full CREATE EXTERNAL TABLE succeeds; DESCRIBE matches"),
    scenario("HIVE-TYPE-DDL", "TC-29", "struct_db.flex_rules_result_full",
             expected_status="VALIDATED", expected_parts=">0", checks=("status", "parts"),
             note="Struct partition column DDL valid (capbusinesseffectivedate)"),
    scenario("HIVE-TYPE-DDL", "TC-30", "struct_db.flex_rules_result_full",
             expected_status="VALIDATED", checks=("status",),
             note="Re-run after a prior failure completes through create_hive_tables"),
    scenario("HIVE-TYPE-DDL", "TC-31", None, expected_status="VALIDATED",
             checks=("no_regression",),
             note="All non-struct tables still succeed — aggregate over every non-struct_db "
                  "DAG-SCENARIO/REGRESSION row in this table"),
    scenario("HIVE-TYPE-DDL", "TC-32", None, phase="MANUAL",
             checks=("manual",),
             blocked_reason="Requires a byte-for-byte diff of generated DDL against a "
                            "pre-fix reference capture, which is not recorded anywhere in "
                            "the repo; TC-31's aggregate check is the automatable part.",
             note="Primitive-only tables' DDL byte-identical to pre-fix output"),

    scenario("EMPTY-PARQUET", "TC-33", "hr_db.employees_parquet_empty",
             expected_status="EMPTY_SOURCE", expected_rows=0,
             expected_columns=TC33_COLUMNS, checks=("status", "rows", "schema_cols"),
             note="EMPTY_SOURCE for Parquet: zero data files, schema read from the "
                  "metastore rather than a file footer, 8-column DDL"),

    scenario("PART-TYPES", "WF-318", "part_types_db.sales_by_date",
             expected_status="VALIDATED", expected_rows=">0",
             checks=("status", "rows", "partition_types"),
             note="DATE + SMALLINT partition keys typed, not STRING"),
    scenario("PART-TYPES", "WF-318-filter", "part_types_db.sales_by_date",
             expected_status="VALIDATED", expected_rows=">0",
             partition_filter="business_date='2024-01-01'", checks=("status", "rows"),
             note="DATE partition filter"),

    scenario("REGRESSION", "R-01", "migration_db.customers", expected_status="VALIDATED",
             expected_rows=10, checks=("status", "rows")),
    scenario("REGRESSION", "R-02", "migration_db.products", expected_status="VALIDATED",
             expected_rows=10, checks=("status", "rows"),
             note="Baseline expectation; P-01 deletes this path in the fault pass"),
    scenario("REGRESSION", "R-03", "migration_db.transactions", expected_status="VALIDATED",
             expected_rows=15, checks=("status", "rows"),
             note="Baseline expectation; P-02 chmods this path in the fault pass"),
    scenario("REGRESSION", "R-04a", "metrics_db.daily_active_users",
             expected_status="VALIDATED", expected_rows=4, checks=("status", "rows")),
    scenario("REGRESSION", "R-04b", "metrics_db.conversion_rates",
             expected_status="VALIDATED", expected_rows=4, checks=("status", "rows")),
    scenario("REGRESSION", "R-04c", "metrics_db.revenue_summary",
             expected_status="VALIDATED", expected_rows=4, checks=("status", "rows")),
    scenario("REGRESSION", "R-05a", "audit_db.access_log", expected_status="VALIDATED",
             expected_rows=5, checks=("status", "rows")),
    scenario("REGRESSION", "R-05b", "audit_db.change_log", expected_status="VALIDATED",
             expected_rows=4, checks=("status", "rows")),
    scenario("REGRESSION", "R-06a", "hr_db_tz.employees", expected_status="VALIDATED",
             expected_rows=6, checks=("status", "rows")),
    scenario("REGRESSION", "R-06b", "hr_db_tz.departments", expected_status="VALIDATED",
             expected_rows=4, checks=("status", "rows")),
    scenario("REGRESSION", "R-07", "sales_db_tz.orders", expected_status="VALIDATED",
             expected_rows=9, expected_parts=3, checks=("status", "rows", "parts")),
    scenario("REGRESSION", "R-08", "sales_db_tz.returns", expected_status="VALIDATED",
             expected_rows=4, expected_parts=3, checks=("status", "rows", "parts")),
    scenario("REGRESSION", "R-09", "sales_db_tz.daily_summary", expected_status="VALIDATED",
             expected_rows=6, expected_parts=3, checks=("status", "rows", "parts")),
    scenario("REGRESSION", "R-10", "analytics_db_tz.events", expected_status="VALIDATED",
             expected_rows=9, expected_parts=3, checks=("status", "rows", "parts")),
    scenario("REGRESSION", "R-11", "logs_db_tz.app_logs", expected_status="VALIDATED",
             expected_rows=10, expected_parts=3, checks=("status", "rows", "parts")),
    scenario("REGRESSION", "R-11-filter", "logs_db_tz.app_logs",
             partition_filter="dt >= '2025-01-15'",
             expected_partition_names={"2025-01-15", "2025-02-01"},
             checks=("partition_value_range",),
             note="[10/11] structural check: SELECT DISTINCT dt WHERE dt >= '2025-01-15' "
                  "must be exactly [2025-01-15, 2025-02-01]. [10/11] asserts the partition "
                  "values, not a row count, so no row count is claimed here."),
    scenario("REGRESSION", "R-12", "edge_cases_db.empty_table", expected_status="EMPTY_SOURCE",
             expected_rows=0, checks=("status", "rows")),
    scenario("REGRESSION", "R-13", "edge_cases_db.nulls_table", expected_status="VALIDATED",
             expected_rows=5, checks=("status", "rows", "nulls_content"),
             note="[10/11] structural check: at least 2 rows with str_col IS NULL"),
    scenario("REGRESSION", "R-14", "edge_cases_db.complex_types", expected_status="VALIDATED",
             expected_rows=3, checks=("status", "rows", "complex_content"),
             note="[10/11] structural check: id=1 has 'spark' in tags and "
                  "address.city = 'San Francisco'"),
    scenario("REGRESSION", "R-15", "edge_cases_db.wide_table", expected_status="VALIDATED",
             expected_rows=5, checks=("status", "rows"), note="50 columns"),
    scenario("REGRESSION", "R-16", "formats_db.orc_table", expected_status="VALIDATED",
             expected_rows=5, checks=("status", "rows")),
    scenario("REGRESSION", "R-17", "formats_db.text_table", expected_status="VALIDATED",
             expected_rows=5, checks=("status", "rows"),
             note="DAG 1 side; the DAG 2 side is P-05 (SKIPPED via INPLACE)"),
    scenario("REGRESSION", "R-18", "formats_db.parquet_table", expected_status="VALIDATED",
             expected_rows=5, checks=("status", "rows")),
    scenario("REGRESSION", "R-19", "analytics_db_tz.sessions", expected_status="VALIDATED",
             expected_rows=6, expected_parts=2, checks=("status", "rows", "parts")),
    scenario("REGRESSION", "R-20a", "tz_db.events_with_ts_la", expected_status="VALIDATED",
             expected_rows=8, checks=("status", "rows")),
    scenario("REGRESSION", "R-20b", "tz_db.orders_with_ts_la", expected_status="VALIDATED",
             expected_rows=8, expected_parts=2, checks=("status", "rows", "parts")),
    scenario("REGRESSION", "R-20c", "tz_db.sessions_with_ts_la", expected_status="VALIDATED",
             expected_rows=6, expected_parts=2, checks=("status", "rows", "parts")),

    scenario("REGRESSION-BASELINE", "BASE-TABLE-NOT-FOUND", "migration_db.nonexistent_tbl",
             expected_status="TABLE_NOT_FOUND", checks=("status",),
             note="[11/11] baseline exception"),
    scenario("REGRESSION-BASELINE", "BASE-DATABASE-NOT-FOUND", "does_not_exist_db.*",
             expected_status="DATABASE_NOT_FOUND", checks=("status_any_in_db",),
             note="[11/11] baseline exception — every token under the missing database"),

    scenario("TRANSIENT-RETRY", "T-01", "retry_test_db.yarn_oom_table", phase="MANUAL",
             expected_status="VALIDATED", expected_rows=20, checks=("rows", "manual"),
             blocked_reason="Needs YARN reconfigured with a low container memory limit "
                            "before DAG 1 runs, which is a cluster-level change this "
                            "notebook must not make. Source row count is still checked.",
             note="YARN OOM during DistCp: run_distcp_ssh retries up to 3 times, log must "
                  "NOT contain [PermanentFail], Airflow attempt count = 3"),
    scenario("TRANSIENT-RETRY", "T-03", None, phase="UNIT", checks=("unit_transient",),
             note="is_permanent_error('distcp', 'Application ... does not exist in the "
                  "list') must be False — YARN AM restart is transient"),
    scenario("TRANSIENT-RETRY", "T-05", None, phase="UNIT", checks=("unit_transient",),
             note="is_permanent_error('iceberg_migrate', 'CommitFailedException: Cannot "
                  "commit due to conflicting transaction') must be False"),

    scenario("PERMANENT-FAIL", "P-01", "migration_db.products", phase="FAULT",
             expected_status="FAILED", expected_attempts=1, log_task="DAG1_DISTCP",
             log_must_contain="[PermanentFail]",
             log_any_of=("Input path does not exist", "File does not exist:"),
             checks=("status", "attempts", "log"), enabled=ENABLE_P01,
             blocked_reason="REGRESSION_ENABLE_P01 is false. This case runs "
                            "hdfs dfs -rm -r on a source table, which cannot be undone "
                            "without re-running setup-test-data.sh, so it is opt-in.",
             note="Source path removed -> permanent_fail on attempt 1, 0 retries."
                  "  DAG-GAP: migration_dag_mapr_to_s3.py never calls permanent_fail(), so no [PermanentFail] marker is logged and retries are not suppressed (default_args retries=2). [11/11] specifies this behaviour but only migration_dag_iceberg.py implements it. Expect this row to FAIL until DAG 1 adopts permanent_fail; that FAIL is the finding, not a broken check."),
    scenario("PERMANENT-FAIL", "P-02", "migration_db.transactions", phase="FAULT",
             expected_status="FAILED", expected_attempts=1, log_task="DAG1_DISTCP",
             log_must_contain="[PermanentFail]", checks=("status", "attempts", "log"),
             enabled=ENABLE_P02,
             blocked_reason="REGRESSION_ENABLE_P02 is false. This case chmod 000s a source "
                            "table; it is restored to 755 afterwards, but it is opt-in.",
             note="Source path chmod 000 -> permanent_fail on attempt 1; restored to 755 "
                  "after.  DAG-GAP: migration_dag_mapr_to_s3.py never calls permanent_fail(), so no [PermanentFail] marker is logged and retries are not suppressed (default_args retries=2). [11/11] specifies this behaviour but only migration_dag_iceberg.py implements it. Expect this row to FAIL until DAG 1 adopts permanent_fail; that FAIL is the finding, not a broken check."),
    scenario("PERMANENT-FAIL", "P-03", f"{s3_dest('migration_db')}.customers", phase="DAG2",
             expected_status="FAILED", expected_attempts=1, log_task="DAG2_MIGRATE",
             log_any_of=("could not read footer", "invalid parquet file"),
             checks=("status", "attempts", "log"), enabled=ENABLE_P03,
             blocked_reason="Requires DAG 2's Excel row for migration_db_s3.customers in "
                            "SNAPSHOT mode. The shipped config has it INPLACE (that is "
                            "P-05's sibling baseline expectation of SKIPPED), and this "
                            "notebook must not modify the Excel configs. Set "
                            "REGRESSION_ENABLE_P03=true once a SNAPSHOT row exists.",
             note="Single corrupt Parquet -> permanent_fail in migrate_tables_to_iceberg"),
    scenario("PERMANENT-FAIL", "P-04a", f"{s3_dest('corrupt_test_db')}.table_a", phase="DAG2",
             expected_status="VALIDATED", checks=("status",), enabled=ENABLE_P04,
             blocked_reason="REGRESSION_ENABLE_P04 is false; the batch corruption is opt-in.",
             note="Multi-table batch: processed BEFORE corrupt_table"),
    scenario("PERMANENT-FAIL", "P-04b", f"{s3_dest('corrupt_test_db')}.corrupt_table", phase="DAG2",
             expected_status="FAILED", checks=("status",), enabled=ENABLE_P04,
             blocked_reason="REGRESSION_ENABLE_P04 is false; the batch corruption is opt-in.",
             note="Multi-table batch: the corrupt table fails permanently"),
    scenario("PERMANENT-FAIL", "P-04c", f"{s3_dest('corrupt_test_db')}.table_c", phase="DAG2",
             expected_status="VALIDATED", checks=("status",), enabled=ENABLE_P04,
             blocked_reason="REGRESSION_ENABLE_P04 is false; the batch corruption is opt-in.",
             note="KEY ASSERTION: processed AFTER corrupt_table and still VALIDATED — "
                  "the _permanent_failure flag must not skip remaining tables"),
    scenario("PERMANENT-FAIL", "P-04-attempts", None, phase="DAG2", expected_attempts=1,
             log_task="DAG2_MIGRATE", checks=("attempts",), enabled=ENABLE_P04,
             blocked_reason="REGRESSION_ENABLE_P04 is false; the batch corruption is opt-in.",
             note="migrate_tables_to_iceberg FAILED permanently with attempt_number = 1 "
                  "(no retries consumed)"),
    scenario("PERMANENT-FAIL", "P-04-source-a", "corrupt_test_db.table_a", phase="DAG1",
             expected_status="VALIDATED", expected_rows=3, checks=("status", "rows"),
             note="[10/11] source row count for the P-04 batch"),
    scenario("PERMANENT-FAIL", "P-04-source-b", "corrupt_test_db.corrupt_table", phase="DAG1",
             expected_status="VALIDATED", expected_rows=2, checks=("status", "rows"),
             note="[10/11] source row count; DAG 1 must succeed for this table before "
                  "the S3 copy is corrupted"),
    scenario("PERMANENT-FAIL", "P-04-source-c", "corrupt_test_db.table_c", phase="DAG1",
             expected_status="VALIDATED", expected_rows=4, checks=("status", "rows"),
             note="[10/11] source row count for the P-04 batch"),
    scenario("PERMANENT-FAIL", "P-05", f"{s3_dest('formats_db')}.text_table", phase="DAG2",
             expected_status="SKIPPED", checks=("status",),
             note="TEXT with INPLACE=TRUE in the DAG 2 Excel -> SKIPPED, not FAILED"),
    scenario("PERMANENT-FAIL", "P-05b", f"{s3_dest('migration_db')}.customers", phase="DAG2",
             expected_status="SKIPPED", checks=("status",), enabled=not ENABLE_P03,
             blocked_reason="Gated off because REGRESSION_ENABLE_P03 is set: P-03 needs the "
                            "same table in SNAPSHOT mode, which contradicts this row's "
                            "INPLACE-derived SKIPPED expectation.",
             note="[11/11] baseline exception: INPLACE on an already-Iceberg table -> "
                  "SKIPPED. Mutually exclusive with P-03."),
    scenario("PERMANENT-FAIL", "P-06", None, phase="EXCEL", expected_attempts=1,
             log_task="DAG1_EXCEL", log_must_contain="[PermanentFail]",
             checks=("task_failed", "attempts", "log"), enabled=ENABLE_P06,
             blocked_reason="REGRESSION_ENABLE_P06 is false; it uploads a garbage object to "
                            "the Excel path and runs DAG 1 against it, so it is opt-in.",
             note="Corrupt Excel -> parse_excel FAILED on attempt 1 with [PermanentFail]. "
                  "The raw failure is BadZipFile: File is not a zip file, which confirms the "
                  "corrupt upload reached parse_excel.  DAG-GAP: migration_dag_mapr_to_s3.py never calls permanent_fail(), so no [PermanentFail] marker is logged and retries are not suppressed (default_args retries=2). [11/11] specifies this behaviour but only migration_dag_iceberg.py implements it. Expect this row to FAIL until DAG 1 adopts permanent_fail; that FAIL is the finding, not a broken check."),
    scenario("PERMANENT-FAIL", "P-07", None, phase="MANUAL", checks=("unit_transient", "manual"),
             blocked_reason="Letting the MapR ticket expire cannot be automated from this "
                            "notebook: the ticket lifetime is a cluster credential, and "
                            "forcing expiry would break every other case in the run. The "
                            "transient-vs-permanent half of the case IS automated below.",
             note="Expired MapR ticket -> validate_prerequisites FAILED on attempt 1 "
                  "(permanent). Compare: is_permanent_error('validate_prerequisites', "
                  "'SSH connection refused') must be False (transient)."),

    scenario("DISTCP-SIZING", "t_tiny", "distcp_sizing_db.t_tiny", expected_status="VALIDATED",
             source_bytes=4096, source_files=1, log_task="DAG1_DISTCP",
             checks=("status", "sizing"), note="min_mappers floor"),
    scenario("DISTCP-SIZING", "t_size_bound", "distcp_sizing_db.t_size_bound",
             expected_status="VALIDATED", source_bytes=6291456, source_files=6,
             log_task="DAG1_DISTCP", checks=("status", "sizing"),
             note="size-driven, no clamp (6 x 1 MiB)"),
    scenario("DISTCP-SIZING", "t_max_clamp", "distcp_sizing_db.t_max_clamp",
             expected_status="VALIDATED", source_bytes=12582912, source_files=12,
             log_task="DAG1_DISTCP", checks=("status", "sizing"),
             note="max_mappers ceiling (12 x 1 MiB)"),
    scenario("DISTCP-SIZING", "t_file_clamp", "distcp_sizing_db.t_file_clamp",
             expected_status="VALIDATED", source_bytes=10485760, source_files=2,
             log_task="DAG1_DISTCP", checks=("status", "sizing"),
             note="file-count clamp, no -blocksperchunk (2 x 5 MiB)"),
    scenario("DISTCP-SIZING", "t_partitioned", "distcp_sizing_db.t_partitioned",
             expected_status="VALIDATED", expected_parts=3,
             partition_filter="dt<='2024-01-02'", source_bytes=7340032, source_files=7,
             log_task="DAG1_DISTCP", checks=("status", "parts", "sizing"),
             note="table-level sizing, filtered to dt=2024-01-01 and dt=2024-01-02"),
    scenario("DISTCP-SIZING", "t_partitioned/dt=2024-01-01",
             "distcp_sizing_db.t_partitioned", expected_status="VALIDATED",
             source_bytes=1048576, source_files=1, log_task="DAG1_DISTCP",
             checks=("sizing",),
             note="per-partition sizing under preserve_delete=true; apportioned bytes = "
                  "7340032 * 1 / 7"),
    scenario("DISTCP-SIZING", "t_partitioned/dt=2024-01-02",
             "distcp_sizing_db.t_partitioned", expected_status="VALIDATED",
             source_bytes=6291456, source_files=6, log_task="DAG1_DISTCP",
             checks=("sizing",),
             note="per-partition sizing under preserve_delete=true; apportioned bytes = "
                  "7340032 * 6 / 7"),
    scenario("DISTCP-SIZING", "t_partitioned/dt=2024-01-03",
             "distcp_sizing_db.t_partitioned", log_task="DAG1_DISTCP",
             checks=("partition_excluded",),
             note="Decoy partition EXCLUDED by the filter: must not appear in any emitted "
                  "distcp command or at the destination"),
    scenario("DISTCP-SIZING", "t_empty", "distcp_sizing_db.t_empty",
             expected_status="EMPTY_SOURCE", expected_rows=0, log_task="DAG1_DISTCP",
             checks=("status", "rows", "no_distcp"),
             note="EMPTY_SOURCE short-circuit — must never reach size_distcp_job"),
]

DAG1_PHASES = ("DAG1", "FAULT", "EXCEL")

if not RUN_DAG1:
    for _row in SCENARIOS:
        if _row["phase"] in DAG1_PHASES and _row["enabled"]:
            _row["enabled"] = False
            _row["blocked_reason"] = ("REGRESSION_RUN_DAG1 is false, so no "
                                      f"{_row['phase']} run happens.")
if not RUN_DAG2:
    for _row in SCENARIOS:
        if _row["phase"] == "DAG2" and _row["enabled"]:
            _row["enabled"] = False
            _row["blocked_reason"] = ("REGRESSION_RUN_DAG2 is false, so no DAG 2 run "
                                      "happens.")

print(f"Scenarios in expected-results table: {len(SCENARIOS)}")
for suite in dict.fromkeys(s["suite"] for s in SCENARIOS):
    rows = [s for s in SCENARIOS if s["suite"] == suite]
    disabled = sum(1 for s in rows if not s["enabled"])
    print(f"  {suite:<21} {len(rows):>3} rows" + (f"  ({disabled} gated off)" if disabled else ""))

### Sizing expectations resolved from the current knobs

Prints what `size_distcp_job` derives for each `DISTCP-SIZING` row at the knob values set in
Step 0, so the table can be eyeballed against the bash `[5/5]` output before any DAG is
triggered. At the documented test knobs this reproduces the `[5/5]` report:
`t_tiny -m 1 -bandwidth 800`, `t_size_bound -m 6 -bandwidth 133`,
`t_max_clamp -m 8 -bandwidth 100`, `t_file_clamp -m 2 -bandwidth 400`,
`t_partitioned -m 7 -bandwidth 114`, and per-partition `dt=2024-01-01 -m 1 -bandwidth 800`,
`dt=2024-01-02 -m 6 -bandwidth 133`.

In [ ]:
print(f"knobs: target={DISTCP_TARGET_BYTES_PER_MAPPER} bytes/mapper  "
      f"min={DISTCP_MIN_MAPPERS}  max={DISTCP_MAX_MAPPERS}  "
      f"aggregate={DISTCP_TARGET_AGGREGATE_MBPS} MB/s\n")
print("{:<34} {:>10} {:>7} {:>8} {:>11}   {}".format(
    "SCENARIO", "BYTES", "FILES", "EXP -m", "EXP -bw", "BRANCH EXERCISED"))
print("-" * 110)
for row in SCENARIOS:
    if row["suite"] != "DISTCP-SIZING":
        continue
    mappers = row["expected_mappers"] if row["expected_mappers"] is not None else "-"
    bandwidth = row["expected_bandwidth"] if row["expected_bandwidth"] is not None else "-"
    print("{:<34} {:>10} {:>7} {:>8} {:>11}   {}".format(
        row["sid"],
        row["source_bytes"] if row["source_bytes"] is not None else "-",
        row["source_files"] if row["source_files"] is not None else "-",
        mappers, bandwidth, row["note"][:44]))

---
## Step 3b: Generate the Excel configs

The DAGs are driven by Excel, so the suite generates both workbooks here from an explicit row
table and uploads them to S3. Generating rather than hand-maintaining them means the config
and the expectations in Step 3 cannot drift apart — and the coverage check below fails loudly
if they do.

### Schemas, taken from the DAGs

Both DAGs read with `pandas.read_excel(engine="openpyxl")` and normalise headers with
`.str.strip().str.lower().str.replace(" ", "_")`, so header case and spacing do not matter.

| DAG | Required columns | Optional |
|---|---|---|
| `source_to_s3_migration` | `database`, `table`, `dest_database`, `bucket` | `partition_filter`, `endpoint` |
| `iceberg_migration` | `database`, `table`, `inplace_migration`, `destination_iceberg_database` | — |

Blank-cell behaviour is the DAGs', not ours: a blank `table` means `*`, a blank
`dest_database` falls back to the source database, a blank `bucket` falls back to
`default_s3_bucket`, and `inplace_migration` accepts `T/TRUE/YES/1` or `F/FALSE/NO/0`.

### Why the row table is separate from the scenario table

Step 3 is flattened to one row per *check*, but several scenarios exist specifically to
exercise Excel **parsing** — `TC-1` a wildcard, `TC-18` a glob, `TC-3`/`TC-22` comma-separated
lists, `TC-21` a wildcard combined with a partition filter. Those cannot be reconstructed from
the flattened table, so the Excel rows are transcribed from the `[6/11]` table in their
original form. The coverage check then verifies every table Step 3 expects is reachable from
some Excel row, expanding `*`, globs and comma lists the way `parse_excel` does.

Part B (`R-xx`, `T-01`, `P-04x`) has no Excel spec in the setup scripts — `[11/11]` only says
those tables should end `VALIDATED`. Those rows use one wildcard row per database with the
`_s3` destination convention the `[11/11]` examples show (`migration_db_s3`,
`formats_db_s3`, `corrupt_test_db_s3`), configurable via `REGRESSION_DEST_DB_SUFFIX`.

### P-03 versus P-05b is resolved here

The conflict from the original build disappears now that the config is generated:
`ENABLE_P03` decides whether `migration_db_s3.customers` is written `inplace_migration=F`
(SNAPSHOT, so P-03's corrupt Parquet makes it `FAILED`) or `T` (INPLACE, so it is `SKIPPED`
per the `[11/11]` baseline). The two can no longer be inconsistent.

### Known gaps, stated rather than hidden

- **`TC-23` (incremental re-run)** is a second DAG run over the same table, not an Excel row.
  The baseline run satisfies its `VALIDATED` check; the incremental behaviour itself is not
  exercised.
- **`TC-19` (alternate bucket)** and **`TC-20` (custom endpoint)** need real values. Set
  `REGRESSION_ALT_BUCKET` / `REGRESSION_CUSTOM_S3_ENDPOINT`; left blank, the rows are written
  with an empty cell and fall back to the default bucket/endpoint, so the scenario is present
  but not genuinely exercised. The cell says so when that happens.
- **`TC-2` and `TC-24`** are the same Excel row (`hr_db / employees`), and `TC-3`'s token list
  is a superset of `TC-2`'s. They resolve to one tracking row; that is inherent to the
  `[6/11]` table, not an artefact of generating it.

In [ ]:
DAG1_EXCEL_COLUMNS = ["database", "table", "dest_database", "bucket",
                      "partition_filter", "endpoint"]
DAG2_EXCEL_COLUMNS = ["database", "table", "inplace_migration",
                      "destination_iceberg_database"]


def dag1_row(database, table, dest_database="", bucket="", partition_filter="",
             endpoint="", note=""):
    return {"database": database, "table": table, "dest_database": dest_database,
            "bucket": bucket, "partition_filter": partition_filter,
            "endpoint": endpoint, "note": note}


def dag2_row(database, table, inplace_migration, destination_iceberg_database="", note=""):
    return {"database": database, "table": table,
            "inplace_migration": "T" if inplace_migration else "F",
            "destination_iceberg_database": destination_iceberg_database, "note": note}


EXCEL_ROWS_DAG1 = [
    dag1_row("sales_db", "*", "sales_db_dest", note="TC-1 wildcard full DB"),
    dag1_row("hr_db", "employees", "hr_db_dest", note="TC-2 single explicit table"),
    dag1_row("hr_db", "employees,departments", "hr_db_dest",
             note="TC-3 comma-separated, one EMPTY_SOURCE"),
    dag1_row("sales_db", "orders", "sales_db_dest",
             partition_filter="year=2024 AND month=1", note="TC-4 partition filter"),
    dag1_row("sales_db", "orders", "sales_db_dest", partition_filter="year=2099",
             note="TC-5 filter matches 0 partitions -> SKIPPED"),
    dag1_row("analytics_db", "sessions", "analytics_db_dest",
             note="TC-6 unregistered partitions -> MSCK REPAIR"),
    dag1_row("analytics_db", "events", "analytics_db_dest", note="TC-7 3-level partitions"),
    dag1_row("analytics_db", "events", "analytics_db_dest",
             partition_filter="region='US' AND year=2024", note="TC-8 filter on 3-level key"),
    dag1_row("logs_db", "error_logs_empty", "logs_db_dest", note="TC-9 EMPTY_SOURCE"),
    dag1_row("logs_db", "metrics_empty_nonpartitioned", "logs_db_dest",
             note="TC-10 EMPTY_SOURCE non-partitioned"),
    dag1_row("analytics_db", "events_empty_partitioned", "analytics_db_dest",
             note="TC-11 registered but empty partitions"),
    dag1_row("analytics_db", "events_parquet_empty_partitioned", "analytics_db_dest",
             note="TC-11b PARQUET variant"),
    dag1_row("sales_db", "orders_empty", "sales_db_dest", note="TC-12 EMPTY_SOURCE"),
    dag1_row("analytics_db", "events_orc", "analytics_db_dest", note="TC-13 ORC"),
    dag1_row("hr_db", "employees_parquet", "hr_db_dest", note="TC-14 PARQUET"),
    dag1_row("hr_db", "employees_avro", "hr_db_dest", note="TC-15 AVRO"),
    dag1_row("logs_db", "app_logs", "logs_db_dest", note="TC-16 TEXTFILE serde"),
    dag1_row("logs_db", "app_logs", "logs_db_dest", partition_filter="dt>='2024-01-15'",
             note="TC-17 string dt filter"),
    dag1_row("sales_db", "orders*", "sales_db_dest", note="TC-18 glob pattern"),
    dag1_row("hr_db", "employees", "hr_renamed", bucket=ALT_BUCKET,
             note="TC-19 different dest_db + alternate bucket"),
    dag1_row("sales_db", "orders", "sales_db_dest", endpoint=CUSTOM_S3_ENDPOINT,
             note="TC-20 custom S3 endpoint"),
    dag1_row("logs_db", "*", "logs_db_dest", partition_filter="dt='2024-01-01'",
             note="TC-21 wildcard + partition filter"),
    dag1_row("analytics_db", "sessions,events", "analytics_db_dest",
             partition_filter="year=2024 AND month=1", note="TC-22 CSV list + filter"),
    dag1_row("hr_db", "employees", "hr_db_dest", note="TC-24 backslash-N NULLs round-trip"),
    dag1_row("sales_db", "orders", "", note="TC-25 blank dest_db defaults to source"),

    dag1_row("struct_db", "*", "struct_db_dest",
             note="HIVE-TYPE-DDL TC-6..TC-30 struct/DDL conversion tables"),
    dag1_row("hr_db", "employees_parquet_empty", "hr_db_dest",
             note="TC-33 empty PARQUET, schema from metastore"),
    dag1_row("part_types_db", "sales_by_date", "part_types_db_dest",
             note="WF-318 DATE + SMALLINT partition keys"),
    dag1_row("part_types_db", "sales_by_date", "part_types_db_dest",
             partition_filter="business_date='2024-01-01'", note="WF-318 DATE filter"),

    dag1_row("migration_db", "customers,products,transactions", s3_dest("migration_db"),
             note="R-01/R-02/R-03; also the P-01 and P-02 fault targets"),
    dag1_row("migration_db", "nonexistent_tbl", s3_dest("migration_db"),
             note="baseline exception: TABLE_NOT_FOUND"),
    dag1_row("does_not_exist_db", "*", s3_dest("does_not_exist_db"),
             note="baseline exception: DATABASE_NOT_FOUND"),
    dag1_row("metrics_db", "*", s3_dest("metrics_db"), note="R-04a/b/c"),
    dag1_row("audit_db", "*", s3_dest("audit_db"), note="R-05a/b"),
    dag1_row("hr_db_tz", "*", s3_dest("hr_db_tz"), note="R-06a/b"),
    dag1_row("sales_db_tz", "*", s3_dest("sales_db_tz"), note="R-07/R-08/R-09"),
    dag1_row("analytics_db_tz", "*", s3_dest("analytics_db_tz"), note="R-10/R-19"),
    dag1_row("logs_db_tz", "*", s3_dest("logs_db_tz"), note="R-11"),
    dag1_row("logs_db_tz", "app_logs", s3_dest("logs_db_tz"),
             partition_filter="dt >= '2025-01-15'", note="R-11 partition filter range"),
    dag1_row("edge_cases_db", "*", s3_dest("edge_cases_db"), note="R-12..R-15"),
    dag1_row("formats_db", "*", s3_dest("formats_db"), note="R-16/R-17/R-18"),
    dag1_row("tz_db", "*", s3_dest("tz_db"), note="R-20a/b/c"),
    dag1_row("retry_test_db", "yarn_oom_table", s3_dest("retry_test_db"),
             note="T-01 source (YARN OOM case is manual)"),
    dag1_row("corrupt_test_db", "*", s3_dest("corrupt_test_db"),
             note="P-04 batch: table_a, corrupt_table, table_c"),

    dag1_row("distcp_sizing_db", "t_tiny,t_size_bound,t_max_clamp,t_file_clamp,t_empty",
             s3_dest("distcp_sizing_db"), note="DISTCP-SIZING non-partitioned fixtures"),
    dag1_row("distcp_sizing_db", "t_partitioned", s3_dest("distcp_sizing_db"),
             partition_filter="dt<='2024-01-02'",
             note="DISTCP-SIZING per-partition; dt=2024-01-03 must stay excluded"),
]

EXCEL_ROWS_DAG2 = [
    dag2_row(s3_dest("migration_db"), "customers", not ENABLE_P03,
             note=("P-03 SNAPSHOT so the corrupt Parquet fails it" if ENABLE_P03
                   else "P-05b INPLACE on already-Iceberg -> SKIPPED")),
    dag2_row(s3_dest("migration_db"), "products,transactions", False, note="R-02/R-03"),
    dag2_row(s3_dest("corrupt_test_db"), "table_a,corrupt_table,table_c", False,
             note="P-04 batch, one corrupt, order matters"),
    dag2_row(s3_dest("formats_db"), "text_table", True,
             note="P-05 TEXT INPLACE -> SKIPPED, not FAILED"),
    dag2_row(s3_dest("formats_db"), "parquet_table,orc_table", False, note="R-16/R-18"),
    dag2_row(s3_dest("metrics_db"), "*", False, note="R-04a/b/c"),
    dag2_row(s3_dest("audit_db"), "*", False, note="R-05a/b"),
    dag2_row(s3_dest("hr_db_tz"), "*", False, note="R-06a/b"),
    dag2_row(s3_dest("sales_db_tz"), "*", False, note="R-07/R-08/R-09"),
    dag2_row(s3_dest("analytics_db_tz"), "*", False, note="R-10/R-19"),
    dag2_row(s3_dest("logs_db_tz"), "*", False, note="R-11"),
    dag2_row(s3_dest("edge_cases_db"), "*", False, note="R-12..R-15"),
    dag2_row(s3_dest("tz_db"), "*", False, note="R-20a/b/c"),
]


def _column_letter(index):
    letters = ""
    while index >= 0:
        letters = chr(ord("A") + index % 26) + letters
        index = index // 26 - 1
    return letters


def _xlsx_via_stdlib(columns, rows):
    """Write a minimal xlsx with zipfile so no third-party writer is needed."""
    import zipfile
    from io import BytesIO
    from xml.sax.saxutils import escape

    def row_xml(number, values):
        cells = "".join(
            f'<c r="{_column_letter(i)}{number}" t="inlineStr">'
            f'<is><t xml:space="preserve">{escape(str(v))}</t></is></c>'
            for i, v in enumerate(values))
        return f'<row r="{number}">{cells}</row>'

    body = row_xml(1, columns) + "".join(
        row_xml(n + 2, [record.get(c, "") for c in columns])
        for n, record in enumerate(rows))

    parts = {
        "[Content_Types].xml":
            '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
            '<Types xmlns="http://schemas.openxmlformats.org/package/2006/content-types">'
            '<Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/>'
            '<Default Extension="xml" ContentType="application/xml"/>'
            '<Override PartName="/xl/workbook.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet.main+xml"/>'
            '<Override PartName="/xl/worksheets/sheet1.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.worksheet+xml"/>'
            '<Override PartName="/xl/styles.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.styles+xml"/>'
            '</Types>',
        "_rels/.rels":
            '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
            '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">'
            '<Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/officeDocument" Target="xl/workbook.xml"/>'
            '</Relationships>',
        "xl/workbook.xml":
            '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
            '<workbook xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main" '
            'xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships">'
            '<sheets><sheet name="Sheet1" sheetId="1" r:id="rId1"/></sheets></workbook>',
        "xl/_rels/workbook.xml.rels":
            '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
            '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">'
            '<Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/worksheet" Target="worksheets/sheet1.xml"/>'
            '<Relationship Id="rId2" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/styles" Target="styles.xml"/>'
            '</Relationships>',
        "xl/styles.xml":
            '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
            '<styleSheet xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main">'
            '<fonts count="1"><font><sz val="11"/><name val="Calibri"/></font></fonts>'
            '<fills count="1"><fill><patternFill patternType="none"/></fill></fills>'
            '<borders count="1"><border/></borders>'
            '<cellStyleXfs count="1"><xf numFmtId="0" fontId="0" fillId="0" borderId="0"/></cellStyleXfs>'
            '<cellXfs count="1"><xf numFmtId="0" fontId="0" fillId="0" borderId="0" xfId="0"/></cellXfs>'
            '<cellStyles count="1"><cellStyle name="Normal" xfId="0" builtinId="0"/></cellStyles>'
            '</styleSheet>',
        "xl/worksheets/sheet1.xml":
            '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
            '<worksheet xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main">'
            f'<sheetData>{body}</sheetData></worksheet>',
    }

    buffer = BytesIO()
    with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as archive:
        for name, payload in parts.items():
            archive.writestr(name, payload)
    return buffer.getvalue()


def build_xlsx_bytes(columns, rows):
    try:
        from io import BytesIO

        from openpyxl import Workbook
    except ImportError:
        return _xlsx_via_stdlib(columns, rows), "stdlib zipfile"
    workbook = Workbook()
    sheet = workbook.active
    sheet.append(columns)
    for record in rows:
        sheet.append([record.get(column, "") for column in columns])
    buffer = BytesIO()
    workbook.save(buffer)
    return buffer.getvalue(), "openpyxl"


def expand_excel_tokens(token):
    return [part.strip() for part in str(token or "*").split(",") if part.strip()]


def excel_covers(rows, database, table):
    """Does any row reach db.table, expanding *, globs and comma lists like parse_excel?"""
    import fnmatch
    for record in rows:
        if record["database"].lower() != database.lower():
            continue
        for token in expand_excel_tokens(record["table"]):
            if token == "*" or fnmatch.fnmatch(table.lower(), token.lower()):
                return True
    return False

### Coverage check, then upload

Every Step 3 scenario that expects a tracking-table row must be reachable from some Excel row.
This is the check that keeps the generated config and the expectations honest: if a scenario
is added to Step 3 without a matching Excel row, it would otherwise silently report
`SKIPPED-NO-ROW` forever, which reads like "not measured" rather than "config is wrong".

Upload only happens when coverage passes.

In [ ]:
import pandas as pd

excel_coverage_gaps = []
for scenario_row in SCENARIOS:
    source = scenario_row["source"]
    if not scenario_row["enabled"] or not source or "." not in source:
        continue
    if "status" not in scenario_row["checks"]:
        continue
    database, table = source.split(".", 1)
    if table == "*":
        continue
    rows = EXCEL_ROWS_DAG2 if scenario_row["phase"] == "DAG2" else EXCEL_ROWS_DAG1
    if not excel_covers(rows, database, table):
        excel_coverage_gaps.append(
            (scenario_row["suite"], scenario_row["sid"], scenario_row["phase"], source))

print(f"DAG 1 Excel rows: {len(EXCEL_ROWS_DAG1)}")
print(f"DAG 2 Excel rows: {len(EXCEL_ROWS_DAG2)}")
if excel_coverage_gaps:
    print(f"\nCOVERAGE GAPS — {len(excel_coverage_gaps)} scenario(s) have no Excel row:")
    for suite, sid, phase, source in excel_coverage_gaps:
        database, table = source.split(".", 1)
        rows = EXCEL_ROWS_DAG2 if phase == "DAG2" else EXCEL_ROWS_DAG1
        print(f"  {suite}/{sid} [{phase}] {source}")
        same_db = [r for r in rows if r["database"].lower() == database.lower()]
        if same_db:
            print(f"      database {database!r} is present; none of its table tokens match "
                  f"{table!r}:")
            for r in same_db:
                print(f"        table={r['table']!r}")
        else:
            near = sorted({r["database"] for r in rows
                           if database.split("_")[0] in r["database"]})
            print(f"      no Excel row has database {database!r}.")
            if near:
                print(f"      similarly-named rows: {near}")
            print(f"      DEST_DB_SUFFIX={DEST_DB_SUFFIX!r} — the scenario table and the "
                  f"Excel rows both derive destination names from it, so a mismatch here "
                  f"usually means stale kernel state. Restart the kernel and run the cells "
                  f"in order.")
else:
    print("\nCoverage OK — every scenario expecting a status row is reachable from the Excel.")

if not ALT_BUCKET:
    print("\nNote: REGRESSION_ALT_BUCKET is unset, so TC-19 falls back to the default bucket "
          "and does not genuinely exercise the alternate-bucket path.")
if not CUSTOM_S3_ENDPOINT:
    print("Note: REGRESSION_CUSTOM_S3_ENDPOINT is unset, so TC-20 falls back to the default "
          "endpoint and does not genuinely exercise the custom-endpoint path.")

display(pd.DataFrame(EXCEL_ROWS_DAG1)[DAG1_EXCEL_COLUMNS + ["note"]])
display(pd.DataFrame(EXCEL_ROWS_DAG2)[DAG2_EXCEL_COLUMNS + ["note"]])

In [ ]:
EXCEL_UPLOADED = False

if not GENERATE_EXCEL_CONFIGS:
    print("REGRESSION_GENERATE_EXCEL=false — using the Excel paths as given, generating nothing.")
elif excel_coverage_gaps:
    print("Refusing to upload: the coverage check above found scenarios with no Excel row. "
          "Fix EXCEL_ROWS_DAG1/EXCEL_ROWS_DAG2 or the scenario table first.")
elif not all(looks_like_s3_uri(p) for p in (DAG1_EXCEL_S3_PATH, DAG2_EXCEL_S3_PATH)):
    print("Refusing to upload: the Excel target paths are not S3 URIs.\n")
    for label, path in (("DAG 1", DAG1_EXCEL_S3_PATH), ("DAG 2", DAG2_EXCEL_S3_PATH)):
        marker = "OK" if looks_like_s3_uri(path) else "NOT A URI"
        print(f"  {label}: {path!r}  [{marker}]")
        if not looks_like_s3_uri(path):
            print(f"    Spark would resolve this relative to its default filesystem, "
                  f"producing a path with the bucket repeated. Use "
                  f"{suggest_s3_uri(path)!r}.")
    print(f"\n  MIGRATION_S3_ROOT  = {MIGRATION_S3_ROOT!r}")
    print(f"  EXCEL_OUTPUT_PREFIX = {EXCEL_OUTPUT_PREFIX!r}")
    print("\n  Fix the root rather than each path: set")
    print(f"    MIGRATION_DEFAULT_S3_BUCKET={suggest_s3_uri(MIGRATION_S3_ROOT)}")
    print("  which also corrects CORRUPT_EXCEL_S3_PATH and the P-03/P-04 destination "
          "prefixes that share the same root.")
else:
    try:
        targets = []
        if RUN_DAG1:
            targets.append((DAG1_EXCEL_S3_PATH, DAG1_EXCEL_COLUMNS, EXCEL_ROWS_DAG1, "DAG 1"))
        if RUN_DAG2:
            targets.append((DAG2_EXCEL_S3_PATH, DAG2_EXCEL_COLUMNS, EXCEL_ROWS_DAG2, "DAG 2"))
        for path, columns, rows, label in targets:
            payload, writer = build_xlsx_bytes(columns, rows)
            s3_write_bytes(path, payload)
            print(f"  {label}: {len(rows)} rows via {writer} -> {path}")
        EXCEL_UPLOADED = True
    except SparkUnavailable as err:
        print(f"Cannot upload without Spark ({err}). The workbooks were not written.")

print(f"\nEXCEL_UPLOADED = {EXCEL_UPLOADED}")

---
## Step 4: Pre-cleanup

Undoes anything a previous run of this notebook could have left behind, so the suite is
idempotent from any prior state:

- Restores `migration_db.db/transactions` to mode 755 in case a previous P-02 pass aborted
  before its restore step.
- Removes the corrupt Excel object P-06 uploads.
- Clears the in-notebook result accumulator.

What this cell deliberately does **not** do is restore data that P-01 destroys or P-03/P-04
corrupt. `hdfs dfs -rm -r` on `migration_db.db/products` and the random bytes written over a
Parquet file are not reversible from here — re-seed with `setup-test-data.sh` (see the final
cell). The cell reports whether those paths look intact so a stale-state run is obvious
instead of silently producing wrong expectations.

In [ ]:
RESULTS.clear()
RUN_IDS.clear()

print("Pre-cleanup: undoing state from prior runs of this notebook...\n")

P01_PATH = edge_hdfs_path("migration_db", "products")
P02_PATH = edge_hdfs_path("migration_db", "transactions")

if EDGE_AVAILABLE:
    restore = edge_shell(f"hdfs dfs -chmod -R 755 {shlex.quote(P02_PATH)}")
    print(f"  chmod 755 {P02_PATH}: exit={restore.returncode}")
    for label, path in (("P-01 source (products)", P01_PATH), ("P-02 source (transactions)", P02_PATH)):
        probe = edge_shell(f"hdfs dfs -test -d {shlex.quote(path)} && echo PRESENT || echo MISSING")
        state = probe.stdout.strip().splitlines()[-1] if probe.stdout.strip() else "UNKNOWN"
        print(f"  {label}: {state}")
        if state != "PRESENT":
            print("    -> re-run setup-test-data.sh before trusting the baseline for this table")
else:
    print("  Edge node unavailable — skipping HDFS restore/probe.")

if CORRUPT_EXCEL_S3_PATH:
    try:
        s3_delete(CORRUPT_EXCEL_S3_PATH)
    except SparkUnavailable as err:
        print(f"  Cannot reach S3 without Spark ({err}) — corrupt Excel object not cleaned up")

print("\nPre-cleanup done.")

---
## Step 5a: Trigger DAG 1 (baseline pass) and wait

Starts `source_to_s3_migration` and polls until it reaches a terminal state. This is the
baseline pass: no faults injected, so every `DAG1`-phase row in the expected-results table
applies to it.

A DAG-level `failed` here is not automatically a suite failure — the baseline set includes
tables that are expected to end `FAILED` / `TABLE_NOT_FOUND` / `DATABASE_NOT_FOUND`, which can
fail the containing task. The per-table verification in Step 6 is what decides.

In [ ]:
DAG1_BASELINE_RUN_ID = None
DAG1_BASELINE_STATE = None

if not RUN_DAG1:
    print("REGRESSION_RUN_DAG1=false — skipping the DAG 1 baseline pass. Every DAG1, FAULT "
          "and EXCEL scenario reports SKIPPED-BY-CONFIG.")
else:
    DAG1_BASELINE_RUN_ID, DAG1_BASELINE_STATE = run_dag_to_completion(
        DAG1_ID, "dag1_baseline")
    print(f"\nDAG 1 baseline run: {DAG1_BASELINE_RUN_ID} -> {DAG1_BASELINE_STATE}")

---
## Step 5b: Inject the P-03 / P-04 Parquet corruption

Sequencing matters and is the whole point of this cell sitting between the two DAG runs:
DAG 1 has just finished copying `corrupt_test_db` (and `migration_db.customers`) to S3, and
DAG 2 has not started yet. `[11/11]` P-04 is explicit about the order — DAG 1 copies all three
tables, then exactly one Parquet file inside `corrupt_table`'s S3 prefix is overwritten with
random bytes, then DAG 2 runs all three as one batch.

`aws s3 cp /dev/urandom ...` in the instructions is replaced by a Hadoop `FileSystem` write of
`os.urandom` bytes over the first `.parquet` object under the prefix — same effect, and it
works through whatever s3a credentials this Spark session already holds instead of needing the
AWS CLI.

P-03 does the same to `migration_db_s3/customers`, and is gated off by default (Step 0
explains why).

If the prefix has no `.parquet` object, the cell says so rather than reporting success.

The destination database names come from `s3_dest()`, the same helper the scenario table and the Excel rows use, so `REGRESSION_DEST_DB_SUFFIX` moves all three together. They were once spelled literally here, which meant a changed suffix made this cell corrupt nothing while P-04b still expected `FAILED` — a confusing failure rather than a clear one.

In [ ]:
def dest_prefix(dest_database, table):
    if MIGRATION_INCLUDE_DB_IN_PATH:
        return f"{MIGRATION_S3_ROOT}/{dest_database}/{table}"
    return f"{MIGRATION_S3_ROOT}/{table}"


def corrupt_one_parquet(dest_database, table, label):
    prefix = dest_prefix(dest_database, table)
    print(f"{label}: prefix {prefix}")
    if not SPARK_AVAILABLE:
        print(f"  Spark unavailable — {label} cannot be injected; its rows will report "
              f"SKIPPED-NO-SPARK rather than a fabricated result")
        return None
    if not s3_exists(prefix):
        print(f"  MISSING — DAG 1 did not produce this prefix; {label} cannot be injected")
        return None
    parquet_files = s3_list_files(prefix, ".parquet")
    if not parquet_files:
        print(f"  No .parquet object under the prefix; {label} cannot be injected")
        return None
    target = parquet_files[0]
    s3_write_bytes(target, os.urandom(4096))
    print(f"  Corrupted: {target}")
    return target


CORRUPTED_OBJECTS = {}

if not RUN_DAG2:
    print("REGRESSION_RUN_DAG2=false — skipping the P-03/P-04 corruption, since nothing "
          "would consume it. Leaving the S3 copies intact.")
elif ENABLE_P04:
    CORRUPTED_OBJECTS["P-04"] = corrupt_one_parquet(
        s3_dest("corrupt_test_db"), "corrupt_table", "P-04")
else:
    print("P-04: disabled via REGRESSION_ENABLE_P04")

if not RUN_DAG2:
    pass
elif ENABLE_P03:
    CORRUPTED_OBJECTS["P-03"] = corrupt_one_parquet(
        s3_dest("migration_db"), "customers", "P-03")
else:
    print("P-03: disabled via REGRESSION_ENABLE_P03 (needs a SNAPSHOT Excel row; see Step 0)")

---
## Step 5c: Trigger DAG 2 and wait

Starts `iceberg_migration` over DAG 1's output, with the P-04 corruption already in place.
`migrate_tables_to_iceberg` is expected to fail permanently on attempt 1 while `table_a` and
`table_c` still reach `VALIDATED` — that is P-04's key assertion, checked in Step 6.

In [ ]:
DAG2_RUN_ID = None
DAG2_STATE = None

if not RUN_DAG2:
    print("REGRESSION_RUN_DAG2=false — skipping DAG 2. Every DAG2 scenario reports "
          "SKIPPED-BY-CONFIG.")
elif not RUN_DAG1 and not SKIP_PREFLIGHT:
    print("REGRESSION_RUN_DAG1=false, so DAG 1 produced no S3 output for DAG 2 to consume "
          "in this session. Skipping DAG 2 — set REGRESSION_SKIP_PREFLIGHT=true to run it "
          "anyway against whatever a previous run left on S3.")
else:
    DAG2_RUN_ID, DAG2_STATE = run_dag_to_completion(DAG2_ID, "dag2_baseline")
    print(f"\nDAG 2 run: {DAG2_RUN_ID} -> {DAG2_STATE}")

---
## Step 5d: Inject P-01 / P-02 and re-run DAG 1 (fault pass)

`[11/11]` puts the permanent-fail HDFS cases after DAG 1 has completed for those tables,
because `R-02` and `R-03` expect the same two tables to be `VALIDATED` in the baseline pass.
So these are a second DAG 1 run, not a modification of the first:

- **P-01** — `hdfs dfs -rm -r` on `migration_db.db/products`. Expected: `run_distcp_ssh`
  `FAILED` on attempt 1, log contains `[PermanentFail]` and either
  `Input path does not exist` or `File does not exist:`.
- **P-02** — `hdfs dfs -chmod 000` on `migration_db.db/transactions`. Expected: permanent fail
  on attempt 1. Restored to 755 in Step 5f.

The re-run is triggered with a `conf` naming the two tables, matching the
"Re-run DAG1 for migration_db/products" instruction. If DAG 1 ignores unrecognised `conf`
keys, the whole DAG re-runs instead of just those tables; that is still a valid fault pass —
the two faulted tables fail and the rest re-validate — only slower. The conf actually sent is
printed so which behaviour occurred is visible in the output.

In [ ]:
FAULT_PASS_RAN = False
DAG1_FAULT_RUN_ID = None
DAG1_FAULT_STATE = None

if not RUN_DAG1:
    print("REGRESSION_RUN_DAG1=false — skipping the fault pass.")
elif not (ENABLE_P01 or ENABLE_P02):
    print("P-01 and P-02 both disabled — skipping the fault pass entirely.")
elif not EDGE_AVAILABLE:
    print("Edge node unavailable — P-01/P-02 cannot be injected. Their rows will report "
          "SKIPPED-NO-EDGE rather than a fabricated result.")
else:
    if ENABLE_P01:
        removal = edge_shell(f"hdfs dfs -rm -r -skipTrash {shlex.quote(P01_PATH)}")
        print(f"P-01: rm -r {P01_PATH} exit={removal.returncode}")
        print(f"  {(removal.stdout or removal.stderr).strip()[:300]}")
    if ENABLE_P02:
        denial = edge_shell(f"hdfs dfs -chmod -R 000 {shlex.quote(P02_PATH)}")
        print(f"P-02: chmod 000 {P02_PATH} exit={denial.returncode}")
        print(f"  {(denial.stdout or denial.stderr).strip()[:300]}")

    fault_conf = {"tables": ["migration_db/products", "migration_db/transactions"]}
    DAG1_FAULT_RUN_ID, DAG1_FAULT_STATE = run_dag_to_completion(
        DAG1_ID, "dag1_fault", conf=fault_conf)
    FAULT_PASS_RAN = True
    print(f"\nDAG 1 fault run: {DAG1_FAULT_RUN_ID} -> {DAG1_FAULT_STATE}")

---
## Step 5e: Inject P-06 and run DAG 1 against the corrupt Excel

A corrupt Excel config fails `parse_excel`, which aborts the whole DAG 1 run before any table
is discovered. That means P-06 cannot share a run with any other case — it gets its own
trigger, last.

Binary garbage is written to `CORRUPT_EXCEL_S3_PATH` and DAG 1 is triggered with
`conf={"excel_file_path": ...}` pointing at it. `excel_file_path` is a declared DAG `Param`
(default `s3a://config-bucket/migration.xlsx`), and the tasks render it as
`{{ params.excel_file_path }}`, so a dag_run conf of the same name overrides it.

Expected: `parse_excel` `FAILED` on attempt 1 with `[PermanentFail]` in the log.

In [ ]:
P06_RUN_ID = None
P06_STATE = None

if not RUN_DAG1:
    print("REGRESSION_RUN_DAG1=false — skipping the P-06 run.")
elif not ENABLE_P06:
    print("P-06: disabled via REGRESSION_ENABLE_P06")
elif not SPARK_AVAILABLE:
    print("P-06: Spark unavailable — cannot upload the corrupt Excel object to S3, so the "
          "run is not triggered. Its rows will report SKIPPED-NO-SPARK.")
else:
    s3_write_bytes(CORRUPT_EXCEL_S3_PATH, os.urandom(8192))
    p06_conf = {"excel_file_path": CORRUPT_EXCEL_S3_PATH}
    P06_RUN_ID, P06_STATE = run_dag_to_completion(DAG1_ID, "dag1_p06", conf=p06_conf)
    print(f"\nDAG 1 P-06 run: {P06_RUN_ID} -> {P06_STATE}")

---
## Step 5f: Restore the P-02 permission change

`[11/11]` P-02 says to restore with `hdfs dfs -chmod 755` after the case. Done here, before
verification, so an aborted verification cell cannot leave the fixture unreadable for the next
run. Runs unconditionally — chmod 755 on an already-755 path is a no-op.

In [ ]:
if EDGE_AVAILABLE:
    restored = edge_shell(f"hdfs dfs -chmod -R 755 {shlex.quote(P02_PATH)}")
    print(f"P-02 restore: chmod 755 {P02_PATH} exit={restored.returncode}")
    verify = edge_shell(f"hdfs dfs -ls -d {shlex.quote(P02_PATH)}")
    print(f"  {verify.stdout.strip()[:300] or verify.stderr.strip()[:300]}")
else:
    print("Edge node unavailable — nothing to restore.")

---
## Step 5g: Cases that are genuinely not automatable

Stated plainly rather than faked or silently dropped. Each still has whatever part of it
*is* automatable wired up, and the report marks the rest `MANUAL`.

**T-01 — YARN OOM during DistCp.** Needs YARN reconfigured with a container memory limit low
enough that the DistCp mappers are killed, before DAG 1 runs. That is a cluster-level
configuration change affecting every other case in the same run, so this notebook does not
make it. Automated here: the `retry_test_db.yarn_oom_table` source row count (20). Manual:
configure YARN, run DAG 1, then confirm `run_distcp_ssh` retried 3 times and the log contains
no `[PermanentFail]`.

**P-07 — expired MapR ticket.** Ticket lifetime is a cluster credential; forcing it to expire
would break the SSH/HDFS access every other case depends on, and there is no way to expire
just one task's ticket. Automated here: the transient-vs-permanent half of the case, i.e.
`is_permanent_error("validate_prerequisites", Exception("SSH connection refused"))` must
return `False`. Manual: let the ticket expire, run DAG 1, confirm `validate_prerequisites`
fails on attempt 1 with no retries.

**P-03 — corrupt Parquet for `migration_db_s3.customers`.** Not an infrastructure limit — a
config conflict. `[11/11]` P-03 requires that table in SNAPSHOT mode, while the shipped DAG 2
Excel has it INPLACE, which is the `[11/11]` baseline's own `SKIPPED` expectation for the same
table (covered as `P-05b`). The two cannot both hold in one run, and this notebook must not
modify the Excel configs. The corruption itself *is* automated in Step 5b — set
`REGRESSION_ENABLE_P03=true` once a SNAPSHOT row exists for it, and disable `P-05b`.

**TC-32 — DDL byte-identical to pre-fix output.** Needs a captured pre-fix DDL baseline to
diff against. No such capture exists in the repo, so there is nothing to compare to. TC-31
(no regression across non-struct tables) is the automatable part and is checked.

---
## Step 5h: T-03 / T-05 / P-07 unit checks

The three `is_permanent_error` cases from `[11/11]` need no infrastructure — they are pure
function calls. `[11/11]` writes them as `from utils.migrations.shared import
is_permanent_error`; the module actually ships as `migrator_utils.migrations.shared`, so both
import paths are tried.

If neither import resolves in this kernel, the checks report `SKIPPED-NO-MODULE` — they are
not silently passed.

In [ ]:
is_permanent_error = None
IMPORT_ERROR = None
for module_path in ("migrator_utils.migrations.shared", "utils.migrations.shared"):
    try:
        module = __import__(module_path, fromlist=["is_permanent_error"])
        is_permanent_error = module.is_permanent_error
        print(f"Imported is_permanent_error from {module_path}")
        break
    except Exception as err:
        IMPORT_ERROR = f"{module_path}: {err}"

if is_permanent_error is None:
    print(f"Could not import is_permanent_error ({IMPORT_ERROR}) — "
          f"T-03 / T-05 / P-07 will report SKIPPED-NO-MODULE")

TRANSIENT_CASES = {
    "T-03": ("distcp",
             "Application application_1234567890_0001 does not exist in the list"),
    "T-05": ("iceberg_migrate",
             "CommitFailedException: Cannot commit due to conflicting transaction"),
    "P-07": ("validate_prerequisites", "SSH connection refused"),
}

TRANSIENT_RESULTS = {}
for case_id, (task_kind, message) in TRANSIENT_CASES.items():
    if is_permanent_error is None:
        TRANSIENT_RESULTS[case_id] = None
        continue
    try:
        TRANSIENT_RESULTS[case_id] = bool(is_permanent_error(task_kind, Exception(message)))
    except Exception as err:
        TRANSIENT_RESULTS[case_id] = f"ERROR: {err}"

for case_id, outcome in TRANSIENT_RESULTS.items():
    task_kind, message = TRANSIENT_CASES[case_id]
    print(f"{case_id}: is_permanent_error({task_kind!r}, {message[:48]!r}...) -> {outcome} "
          f"(expected False)")

---
## Step 6: Verification

Walks the expected-results table and, per row, gathers the actual value for every check it
declares. Four sources are consulted:

1. **Tracking tables** — `migration_table_status` for `DAG1` / `FAULT` rows and
   `iceberg_migration_table_status` for `DAG2` rows, read with `spark.sql` and keyed by
   `(source_database, source_table)`. DAG 2's column names are resolved from the DataFrame
   schema against a candidate list rather than assumed, so a rename there degrades to a clear
   `ERROR` instead of a wrong answer.
2. **Source side over the edge node** — row counts, `SHOW PARTITIONS`, `DESCRIBE`, and the
   struct/partition-type/content assertions, reusing the exact query shapes from `[5/11]` and
   `[10/11]`: `SELECT COUNT(*)`, `SHOW PARTITIONS`, `DESCRIBE`, the
   `(?<=\w):(?=\w)` regex applied twice for idempotency, and the DATE/SMALLINT partition-type
   parse out of the `# Partition Information` block.
3. **Airflow task instances** — `try_number` for the attempt-count assertions.
4. **Airflow task logs** — `-m` / `-bandwidth` for the sizing rows, `[PermanentFail]` and the
   error substrings for P-01/P-02/P-03/P-06, and absence of any distcp command for the
   excluded partition and the empty table.

A row whose prerequisite is unavailable is recorded as `SKIPPED-*` with the reason, never as a
pass. Logs are fetched once per task and cached.

In [ ]:
STATUS_DB_COLUMNS     = ("source_database", "database_name", "db_name", "source_db")
STATUS_TABLE_COLUMNS  = ("source_table", "table_name", "tbl_name", "source_tbl")
STATUS_VALUE_COLUMNS  = ("overall_status", "status", "migration_status")
STATUS_DEST_COLUMNS   = ("dest_database", "destination_iceberg_database", "dest_db")
STATUS_FILTER_COLUMNS = ("partition_filter",)

LOG_TASKS = {
    "DAG1_DISTCP": (DAG1_ID, "DAG1_BASELINE", DAG1_DISTCP_TASK_ID),
    "DAG1_FAULT_DISTCP": (DAG1_ID, "DAG1_FAULT", DAG1_DISTCP_TASK_ID),
    "DAG2_MIGRATE": (DAG2_ID, "DAG2", DAG2_MIGRATE_TASK_ID),
    "DAG1_EXCEL": (DAG1_ID, "P06", DAG1_EXCEL_TASK_ID),
}

RUN_HANDLES = {
    "DAG1_BASELINE": DAG1_BASELINE_RUN_ID,
    "DAG1_FAULT": DAG1_FAULT_RUN_ID,
    "DAG2": DAG2_RUN_ID,
    "P06": P06_RUN_ID,
}

PHASE_TO_RUN = {
    "DAG1": ("DAG1_BASELINE", DAG1_STATUS_TABLE, f"{TRACKING_DB}.migration_runs"),
    "FAULT": ("DAG1_FAULT", DAG1_STATUS_TABLE, f"{TRACKING_DB}.migration_runs"),
    "DAG2": ("DAG2", DAG2_STATUS_TABLE, f"{TRACKING_DB}.iceberg_migration_runs"),
    "EXCEL": ("P06", DAG1_STATUS_TABLE, f"{TRACKING_DB}.migration_runs"),
}

_log_cache = {}
_status_cache = {}
_run_id_cache = {}


def _pick_column(available, candidates):
    lowered = {name.lower(): name for name in available}
    for candidate in candidates:
        if candidate in lowered:
            return lowered[candidate]
    return None


def _normalise_filter(value):
    """Compare partition filters ignoring whitespace and case.

    The tracking row stores the filter as the Excel supplied it; the scenario table
    spells it the same way but small spacing differences must not break the match.
    """
    if value is None:
        return ""
    collapsed = " ".join(str(value).split())
    return "" if collapsed.lower() in ("", "nan", "none") else collapsed.lower()


class TrackingRunNotFound(RuntimeError):
    pass


def resolve_tracking_run_id(runs_table, dag_run_id):
    """Map an Airflow dag_run_id to the DAG's own run_id.

    The tracking tables are keyed by the run_id that create_migration_run generates, not
    by the Airflow dag_run_id, and several runs of the same DAG accumulate in the same
    table. Selecting 'the most recent run_id' is wrong as soon as the fault pass exists:
    the baseline checks would silently read the fault pass's rows. migration_runs carries
    both ids, so the mapping is exact.
    """
    key = (runs_table, dag_run_id)
    if key in _run_id_cache:
        return _run_id_cache[key]
    session = require_spark()
    try:
        rows = session.sql(
            f"SELECT run_id FROM {runs_table} WHERE dag_run_id = '{dag_run_id}' "
            f"ORDER BY started_at DESC LIMIT 1").collect()
    except SparkUnavailable:
        raise
    except Exception as err:
        raise TrackingRunNotFound(
            f"could not read {runs_table} to map dag_run_id {dag_run_id!r}: {err}") from None
    if not rows:
        raise TrackingRunNotFound(
            f"{runs_table} has no row with dag_run_id={dag_run_id!r}. The DAG run did not "
            f"reach create_migration_run, so no tracking rows exist for it.")
    resolved = rows[0]["run_id"]
    _run_id_cache[key] = resolved
    return resolved


def load_status_rows(status_table, runs_table, dag_run_id):
    """Index one DAG run's tracking rows by (database, table) -> list of candidates.

    A single table appears more than once in one run when the Excel drives it with
    different partition filters or destinations (TC-4 vs TC-5, TC-2 vs TC-19,
    TC-1a vs TC-25). Keying by (database, table) alone collapses those to one arbitrary
    row, so each candidate keeps its dest_database and partition_filter for matching.
    """
    key = (status_table, dag_run_id)
    if key in _status_cache:
        return _status_cache[key]

    run_id = resolve_tracking_run_id(runs_table, dag_run_id)
    frame = require_spark().sql(f"SELECT * FROM {status_table}")
    columns = frame.columns
    db_col = _pick_column(columns, STATUS_DB_COLUMNS)
    tbl_col = _pick_column(columns, STATUS_TABLE_COLUMNS)
    status_col = _pick_column(columns, STATUS_VALUE_COLUMNS)
    if not (db_col and tbl_col and status_col):
        raise RuntimeError(
            f"{status_table} has no recognisable db/table/status columns; saw {columns}")
    dest_col = _pick_column(columns, STATUS_DEST_COLUMNS)
    filter_col = _pick_column(columns, STATUS_FILTER_COLUMNS)
    run_col = _pick_column(columns, ("run_id",))
    if run_col:
        frame = frame.where(f"{run_col} = '{run_id}'")

    index = {}
    for record in frame.collect():
        as_dict = record.asDict()
        candidate = {
            "overall_status": as_dict[status_col],
            "dest_database": (str(as_dict[dest_col]) if dest_col
                              and as_dict.get(dest_col) is not None else None),
            "partition_filter": _normalise_filter(
                as_dict.get(filter_col) if filter_col else None),
            "row": as_dict,
        }
        index.setdefault(
            (str(as_dict[db_col]).lower(), str(as_dict[tbl_col]).lower()), []
        ).append(candidate)

    resolved = {"rows": index, "run_id": run_id, "status_column": status_col,
                "has_filter_column": filter_col is not None,
                "has_dest_column": dest_col is not None}
    _status_cache[key] = resolved
    total = sum(len(v) for v in index.values())
    print(f"  {status_table}: {total} row(s) across {len(index)} table(s) for "
          f"dag_run_id={dag_run_id} -> run_id={run_id}")
    return resolved


class AmbiguousStatusRow(RuntimeError):
    pass


def status_for(scenario_row):
    """Find the one tracking row a scenario refers to, or say precisely why it cannot."""
    mapped = PHASE_TO_RUN.get(scenario_row["phase"])
    if not mapped:
        return None, f"phase {scenario_row['phase']} has no DAG run"
    handle, status_table, runs_table = mapped
    dag_run_id = RUN_HANDLES.get(handle)
    if dag_run_id is None:
        return None, f"{handle} run did not execute"
    try:
        loaded = load_status_rows(status_table, runs_table, dag_run_id)
    except (SparkUnavailable, TrackingRunNotFound):
        raise
    except Exception as err:
        return None, f"status table unreadable: {err}"

    source = scenario_row["source"] or ""
    if "." not in source:
        return None, "no source table on this row"
    database, table = source.split(".", 1)
    candidates = loaded["rows"].get((database.lower(), table.lower()), [])
    if not candidates:
        return None, f"no row for {source} in {status_table} (run_id={loaded['run_id']})"

    wanted_filter = _normalise_filter(scenario_row["partition_filter"])
    if loaded["has_filter_column"]:
        narrowed = [c for c in candidates if c["partition_filter"] == wanted_filter]
        if not narrowed:
            seen = sorted({c["partition_filter"] or "(none)" for c in candidates})
            return None, (f"{source} has no row with partition_filter="
                          f"{wanted_filter or '(none)'!r}; run has {seen}")
        candidates = narrowed

    wanted_dest = scenario_row.get("dest_database")
    if wanted_dest and loaded["has_dest_column"]:
        narrowed = [c for c in candidates
                    if (c["dest_database"] or "").lower() == wanted_dest.lower()]
        if not narrowed:
            seen = sorted({c["dest_database"] or "(none)" for c in candidates})
            return None, (f"{source} has no row with dest_database={wanted_dest!r}; "
                          f"run has {seen}")
        candidates = narrowed

    if len(candidates) > 1:
        distinct = {(c["dest_database"], c["partition_filter"], c["overall_status"])
                    for c in candidates}
        if len(distinct) > 1:
            raise AmbiguousStatusRow(
                f"{source} matches {len(candidates)} tracking rows that differ: "
                f"{sorted(distinct)}. Add dest_database to this scenario to disambiguate.")
    return candidates[0], None


COLON_RE = re.compile(r"(?<=\w):(?=\w)")


def struct_ddl_verdict(table, column):
    described = dict(edge_describe(table))
    type_string = (described.get(column) or "").strip()
    if not type_string:
        return "FAIL", f"column {column} not found in DESCRIBE {table}"
    once = COLON_RE.sub(" ", type_string)
    twice = COLON_RE.sub(" ", once)
    colons_before = len(COLON_RE.findall(type_string))
    colons_after = len(COLON_RE.findall(once))
    idempotent = once == twice
    verdict = "PASS" if colons_after == 0 and idempotent else "FAIL"
    return verdict, f"colons {colons_before}->{colons_after}, idempotent={idempotent}"


def partition_types_verdict(table):
    described = edge_describe(table)
    part_types = {}
    in_partition_block = False
    for name, dtype in described:
        if name == "# Partition Information":
            in_partition_block = True
            continue
        if in_partition_block and (name == "# col_name" or not name):
            continue
        if in_partition_block and name.startswith("#"):
            break
        if in_partition_block and name:
            part_types[name] = dtype
    expected = {"business_date": "date", "cyc_num": "smallint"}
    mismatches = {k: (v, part_types.get(k)) for k, v in expected.items()
                  if (part_types.get(k) or "").lower() != v}
    return ("PASS" if not mismatches else "FAIL"), f"{part_types or mismatches}"


def cached_log(log_key):
    if log_key in _log_cache:
        return _log_cache[log_key]
    if log_key not in LOG_TASKS:
        _log_cache[log_key] = ""
        return ""
    dag_id, handle, task_id = LOG_TASKS[log_key]
    dag_run_id = RUN_HANDLES.get(handle)
    text = "" if dag_run_id is None else all_task_logs(dag_id, dag_run_id, task_id)
    _log_cache[log_key] = text
    return text


def cached_task_instance(log_key):
    if log_key not in LOG_TASKS:
        return None
    dag_id, handle, task_id = LOG_TASKS[log_key]
    dag_run_id = RUN_HANDLES.get(handle)
    if dag_run_id is None:
        return None
    return task_instance(dag_id, dag_run_id, task_id)


def cached_max_try_number(log_key):
    """Attempt count across every mapped instance, not just the first one."""
    if log_key not in LOG_TASKS:
        return None
    dag_id, handle, task_id = LOG_TASKS[log_key]
    dag_run_id = RUN_HANDLES.get(handle)
    if dag_run_id is None:
        return None
    return max_try_number(dag_id, dag_run_id, task_id)


def cached_task_states(log_key):
    """(state, map_index) for every instance — an expanded task fails per index."""
    if log_key not in LOG_TASKS:
        return []
    dag_id, handle, task_id = LOG_TASKS[log_key]
    dag_run_id = RUN_HANDLES.get(handle)
    if dag_run_id is None:
        return []
    return [(ti.get("state"), ti.get("map_index"))
            for ti in task_instances_for(dag_id, dag_run_id, task_id)]


def effective_log_key(scenario_row):
    key = scenario_row["log_task"]
    if key == "DAG1_DISTCP" and scenario_row["phase"] == "FAULT":
        return "DAG1_FAULT_DISTCP"
    return key


def record(scenario_row, check, expected, actual, verdict, detail=""):
    RESULTS.append({
        "suite": scenario_row["suite"], "id": scenario_row["sid"],
        "source": scenario_row["source"] or "-", "phase": scenario_row["phase"],
        "check": check, "expected": expected, "actual": actual,
        "result": verdict, "detail": detail or scenario_row["note"],
    })


def compare(scenario_row, check, expected, actual):
    if expected == ">0":
        verdict = "PASS" if isinstance(actual, int) and actual > 0 else "FAIL"
    else:
        verdict = "PASS" if actual == expected else "FAIL"
    record(scenario_row, check, expected, actual, verdict)


print("Loading tracking tables...")
if not SPARK_AVAILABLE:
    print(f"  Spark unavailable ({SPARK_IMPORT_ERROR}) — every overall_status check will "
          f"report SKIPPED-NO-SPARK")
for _phase_name, (handle, status_table, runs_table) in PHASE_TO_RUN.items():
    if RUN_HANDLES.get(handle):
        try:
            load_status_rows(status_table, runs_table, RUN_HANDLES[handle])
        except Exception as err:
            print(f"  {status_table} ({handle}): {err}")
print()

### Run every declared check

One pass over the expected-results table. Each check kind maps to exactly one comparison, and
anything that raises is recorded as `ERROR` against that row rather than aborting the pass, so
a single broken fixture does not hide the other 100+ results.

In [ ]:
NON_STRUCT_SUITES = ("DAG-SCENARIO", "REGRESSION", "PART-TYPES", "EMPTY-PARQUET")

for scenario_row in SCENARIOS:
    checks = scenario_row["checks"]
    source = scenario_row["source"]

    if not scenario_row["enabled"]:
        record(scenario_row, "gated", scenario_row["expected_status"], "not run",
               "SKIPPED-BY-CONFIG", scenario_row["blocked_reason"] or scenario_row["note"])
        continue

    for check in checks:
        try:
            if check == "status":
                found, reason = status_for(scenario_row)
                if found is None:
                    record(scenario_row, "overall_status", scenario_row["expected_status"],
                           "-", "SKIPPED-NO-ROW", reason)
                else:
                    compare(scenario_row, "overall_status", scenario_row["expected_status"],
                            found["overall_status"])

            elif check == "status_any_in_db":
                handle, status_table, runs_table = PHASE_TO_RUN[scenario_row["phase"]]
                loaded = load_status_rows(status_table, runs_table, RUN_HANDLES.get(handle))
                database = (source or "").split(".", 1)[0].lower()
                observed = {candidate["overall_status"]
                            for (db, _t), candidates in loaded["rows"].items()
                            if db == database for candidate in candidates}
                verdict = ("PASS" if observed and observed == {scenario_row["expected_status"]}
                           else "FAIL")
                record(scenario_row, "overall_status (all tokens)",
                       scenario_row["expected_status"], sorted(observed) or "-", verdict)

            elif check == "rows":
                if not EDGE_AVAILABLE:
                    record(scenario_row, "source_row_count", scenario_row["expected_rows"],
                           "-", "SKIPPED-NO-EDGE", "edge node not reachable")
                else:
                    compare(scenario_row, "source_row_count", scenario_row["expected_rows"],
                            edge_row_count(source, scenario_row["partition_filter"]))

            elif check == "parts":
                if not EDGE_AVAILABLE:
                    record(scenario_row, "source_partition_count",
                           scenario_row["expected_parts"], "-", "SKIPPED-NO-EDGE",
                           "edge node not reachable")
                else:
                    compare(scenario_row, "source_partition_count",
                            scenario_row["expected_parts"], edge_partition_count(source))

            elif check == "partition_names":
                if not EDGE_AVAILABLE:
                    record(scenario_row, "partition_names", "see note", "-",
                           "SKIPPED-NO-EDGE", "edge node not reachable")
                else:
                    found_names = edge_partition_names(source)
                    expected_names = scenario_row["expected_partition_names"]
                    verdict = "PASS" if found_names == expected_names else "FAIL"
                    record(scenario_row, "partition_names", sorted(expected_names),
                           sorted(found_names), verdict)

            elif check == "schema_cols":
                if not EDGE_AVAILABLE:
                    record(scenario_row, "schema_columns", scenario_row["expected_columns"],
                           "-", "SKIPPED-NO-EDGE", "edge node not reachable")
                else:
                    described = [name for name, _ in edge_describe(source)
                                 if name and not name.startswith("#")]
                    expected_columns = scenario_row["expected_columns"]
                    verdict = "PASS" if described[:len(expected_columns)] == expected_columns \
                        and not set(described) - set(expected_columns) else "FAIL"
                    record(scenario_row, "schema_columns", expected_columns, described, verdict)

            elif check == "struct_ddl":
                if not EDGE_AVAILABLE:
                    record(scenario_row, "struct_ddl_colons", "0 colons, idempotent", "-",
                           "SKIPPED-NO-EDGE", "edge node not reachable")
                else:
                    verdict, detail = struct_ddl_verdict(source, scenario_row["struct_column"])
                    record(scenario_row, f"struct_ddl_colons[{scenario_row['struct_column']}]",
                           "0 colons, idempotent", detail, verdict, detail)

            elif check == "partition_types":
                if not EDGE_AVAILABLE:
                    record(scenario_row, "partition_column_types",
                           "business_date=date, cyc_num=smallint", "-", "SKIPPED-NO-EDGE",
                           "edge node not reachable")
                else:
                    verdict, detail = partition_types_verdict(source)
                    record(scenario_row, "partition_column_types",
                           "business_date=date, cyc_num=smallint", detail, verdict, detail)

            elif check == "partition_value_range":
                if not EDGE_AVAILABLE:
                    record(scenario_row, "distinct_partition_values",
                           sorted(scenario_row["expected_partition_names"]), "-",
                           "SKIPPED-NO-EDGE", "edge node not reachable")
                else:
                    column = (scenario_row["partition_filter"] or "").split()[0]
                    rows = edge_sql(f"SELECT DISTINCT {column} FROM {source} "
                                    f"WHERE {scenario_row['partition_filter']} "
                                    f"ORDER BY {column}")
                    found = {str(list(r.values())[0]).strip() for r in rows}
                    expected_values = scenario_row["expected_partition_names"]
                    record(scenario_row, f"distinct {column} in range",
                           sorted(expected_values), sorted(found),
                           "PASS" if found == expected_values else "FAIL")

            elif check == "nulls_content":
                if not EDGE_AVAILABLE:
                    record(scenario_row, "null_str_col_rows", ">=2", "-", "SKIPPED-NO-EDGE",
                           "edge node not reachable")
                else:
                    null_rows = edge_row_count(source, "str_col IS NULL")
                    record(scenario_row, "null_str_col_rows", ">=2", null_rows,
                           "PASS" if null_rows >= 2 else "FAIL")

            elif check == "complex_content":
                if not EDGE_AVAILABLE:
                    record(scenario_row, "struct_array_content",
                           "tags contains 'spark', address.city='San Francisco'", "-",
                           "SKIPPED-NO-EDGE", "edge node not reachable")
                else:
                    rows = edge_sql(f"SELECT tags, address FROM {source} WHERE id = 1")
                    blob = json.dumps(rows)
                    verdict = ("PASS" if "spark" in blob and "San Francisco" in blob else "FAIL")
                    record(scenario_row, "struct_array_content",
                           "tags contains 'spark', address.city='San Francisco'",
                           blob[:120], verdict)

            elif check == "attempts":
                attempts = cached_max_try_number(effective_log_key(scenario_row))
                if attempts is None:
                    record(scenario_row, "airflow_attempt_number",
                           scenario_row["expected_attempts"], "-", "SKIPPED-NO-RUN",
                           "no task instance for this task in that run")
                else:
                    compare(scenario_row, "max_attempt_over_mapped_instances",
                            scenario_row["expected_attempts"], attempts)

            elif check == "task_failed":
                states = cached_task_states(effective_log_key(scenario_row))
                if not states:
                    record(scenario_row, "task_state", "failed", "-", "SKIPPED-NO-RUN",
                           "no task instance for this task in that run")
                else:
                    observed = [s for s, _ in states]
                    record(scenario_row, "task_state", "at least one failed",
                           observed, "PASS" if "failed" in observed else "FAIL")

            elif check == "log":
                text = cached_log(effective_log_key(scenario_row))
                if not text:
                    record(scenario_row, "task_log", "log markers present", "-",
                           "SKIPPED-NO-LOG", "no log content retrieved")
                else:
                    findings = []
                    verdict = "PASS"
                    if scenario_row["log_must_contain"]:
                        present = scenario_row["log_must_contain"] in text
                        findings.append(f"{scenario_row['log_must_contain']}={present}")
                        verdict = verdict if present else "FAIL"
                    if scenario_row["log_must_not_contain"]:
                        absent = scenario_row["log_must_not_contain"] not in text
                        findings.append(f"no {scenario_row['log_must_not_contain']}={absent}")
                        verdict = verdict if absent else "FAIL"
                    if scenario_row["log_any_of"]:
                        hits = [needle for needle in scenario_row["log_any_of"] if needle in text]
                        findings.append(f"any_of hits={hits}")
                        verdict = verdict if hits else "FAIL"
                    record(scenario_row, "task_log", "markers present",
                           "; ".join(findings), verdict)

            elif check == "sizing":
                text = cached_log(effective_log_key(scenario_row))
                expected_pair = (scenario_row["expected_mappers"],
                                 scenario_row["expected_bandwidth"])
                is_partition = "/" in scenario_row["sid"]
                if not text:
                    record(scenario_row, "distcp sizing", expected_pair, "-",
                           "SKIPPED-NO-LOG", "no log content retrieved")
                elif is_partition:
                    partition = scenario_row["sid"].split("/", 1)[1]
                    pairs = distcp_sizing_for_partition(text, partition)
                    if not pairs:
                        record(scenario_row, f"distcp sizing [{partition}]", expected_pair,
                               "no matching line", "SKIPPED-NO-LOG",
                               "the per-partition line is emitted with logger.debug, so it "
                               "is absent unless the DAG log level is DEBUG. Absence here "
                               "means not logged, not mis-sized.")
                    else:
                        record(scenario_row, f"distcp sizing [{partition}]", expected_pair,
                               pairs, "PASS" if expected_pair in pairs else "FAIL")
                else:
                    pairs = distcp_sizing_for_table(text, source)
                    record(scenario_row, f"distcp sizing [{source}]", expected_pair,
                           pairs or "no '[DistCp] Sized' line for this table",
                           "PASS" if expected_pair in pairs else "FAIL")

            elif check == "partition_excluded":
                text = cached_log(effective_log_key(scenario_row))
                excluded = scenario_row["sid"].split("/")[-1]
                if not text:
                    record(scenario_row, "excluded_partition_absent", f"no {excluded}", "-",
                           "SKIPPED-NO-LOG", "no log content retrieved")
                else:
                    commands = distcp_commands_for_path(text, excluded)
                    sized = distcp_sizing_for_partition(text, excluded)
                    record(scenario_row, "excluded_partition_absent",
                           f"{excluded} in 0 distcp commands",
                           f"{len(commands)} command(s), {len(sized)} sizing line(s)",
                           "PASS" if not commands and not sized else "FAIL")

            elif check == "no_distcp":
                text = cached_log(effective_log_key(scenario_row))
                table_name = (source or "").split(".")[-1]
                if not text:
                    record(scenario_row, "no_distcp_for_empty", f"no distcp for {table_name}",
                           "-", "SKIPPED-NO-LOG", "no log content retrieved")
                else:
                    commands = distcp_commands_for_path(text, f"/{table_name}")
                    marked = empty_source_logged(text, source)
                    findings = (f"EMPTY_SOURCE logged={marked}, "
                                f"{len(commands)} distcp command(s)")
                    record(scenario_row, "no_distcp_for_empty",
                           "EMPTY_SOURCE logged and 0 distcp commands", findings,
                           "PASS" if marked and not commands else "FAIL")

            elif check == "unit_transient":
                outcome = TRANSIENT_RESULTS.get(scenario_row["sid"])
                if outcome is None:
                    record(scenario_row, "is_permanent_error", False, "-",
                           "SKIPPED-NO-MODULE", IMPORT_ERROR or "module not importable")
                else:
                    record(scenario_row, "is_permanent_error", False, outcome,
                           "PASS" if outcome is False else "FAIL")

            elif check == "manual":
                record(scenario_row, "manual", scenario_row["expected_status"] or "see note",
                       "requires an operator step", "MANUAL",
                       scenario_row["blocked_reason"] or scenario_row["note"])

        except AmbiguousStatusRow as err:
            record(scenario_row, check, scenario_row["expected_status"] or "-",
                   "ambiguous", "ERROR", str(err))
        except TrackingRunNotFound as err:
            record(scenario_row, check, scenario_row["expected_status"] or "-", "-",
                   "SKIPPED-NO-ROW", str(err))
        except SparkUnavailable as err:
            record(scenario_row, check, scenario_row["expected_status"] or "-", "-",
                   "SKIPPED-NO-SPARK", f"needs a PySpark kernel: {err}")
        except EdgeUnavailable as err:
            record(scenario_row, check, "-", "-", "SKIPPED-NO-EDGE", str(err))
        except Exception as err:
            record(scenario_row, check, "-", f"{type(err).__name__}: {err}", "ERROR",
                   str(err)[:200])

for scenario_row in SCENARIOS:
    if "no_regression" not in scenario_row["checks"] or not scenario_row["enabled"]:
        continue
    relevant = [r for r in RESULTS
                if r["check"] == "overall_status" and r["suite"] in NON_STRUCT_SUITES]
    failures = [r["id"] for r in relevant if r["result"] == "FAIL"]
    if not relevant:
        record(scenario_row, "non_struct_tables_still_pass", "0 overall_status failures",
               "no overall_status results to aggregate", "SKIPPED-NO-ROW",
               "nothing ran that this could summarise")
    else:
        record(scenario_row, "non_struct_tables_still_pass",
               f"0 of {len(relevant)} overall_status failures",
               f"{len(failures)} failure(s): {failures[:8]}",
               "PASS" if not failures else "FAIL")

print(f"Recorded {len(RESULTS)} check results across {len(SCENARIOS)} scenarios.")

---
## Step 7: Report

One row per check: suite, scenario id, source table, phase, check, expected, actual, result.
Rendered as a pandas DataFrame so it reads as a table in the notebook, followed by a printed
summary.

`PASS` / `FAIL` are the verdicts that matter. `MANUAL` and `SKIPPED-*` are neither — they are
counted and listed separately so a run that silently lost its edge node or Airflow logs cannot
be mistaken for a green run. The overall line is `REGRESSION PASSED` only when there are zero
`FAIL` and zero `ERROR` rows.

In [ ]:
import pandas as pd

pd.set_option("display.max_rows", 400)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 240)

report = pd.DataFrame(RESULTS, columns=[
    "suite", "id", "source", "phase", "check", "expected", "actual", "result", "detail"])

RESULT_ORDER = {"FAIL": 0, "ERROR": 1, "MANUAL": 2, "SKIPPED-BY-CONFIG": 3,
                "SKIPPED-NO-SPARK": 4, "SKIPPED-NO-EDGE": 5, "SKIPPED-NO-ROW": 6,
                "SKIPPED-NO-LOG": 7, "SKIPPED-NO-RUN": 8, "SKIPPED-NO-MODULE": 9,
                "PASS": 10}
report = report.assign(_order=report["result"].map(RESULT_ORDER).fillna(99)) \
               .sort_values(["_order", "suite", "id", "check"]) \
               .drop(columns="_order") \
               .reset_index(drop=True)

display(report.drop(columns="detail"))

### Summary

Counts by verdict, a per-suite breakdown, the detail text for everything that is not a plain
`PASS`, and the overall regression line.

In [ ]:
counts = report["result"].value_counts().to_dict()
passed = counts.get("PASS", 0)
failed = counts.get("FAIL", 0)
errored = counts.get("ERROR", 0)
manual = counts.get("MANUAL", 0)
skipped = sum(v for k, v in counts.items() if k.startswith("SKIPPED"))

print("Verdict counts")
print("-" * 60)
for verdict in sorted(counts, key=lambda k: RESULT_ORDER.get(k, 99)):
    print(f"  {verdict:<20} {counts[verdict]:>4}")

print("\nPer suite")
print("-" * 60)
by_suite = report.pivot_table(index="suite", columns="result", values="id",
                              aggfunc="count", fill_value=0)
print(by_suite.to_string())

needs_attention = report[report["result"] != "PASS"]
if not needs_attention.empty:
    print("\nEverything that is not a plain PASS")
    print("-" * 60)
    for _, row in needs_attention.iterrows():
        print(f"  [{row['result']}] {row['suite']}/{row['id']} :: {row['check']}")
        print(f"      expected={row['expected']!r}")
        print(f"      actual  ={row['actual']!r}")
        if row["detail"]:
            print(f"      {row['detail']}")

print("\nDAG runs")
print("-" * 60)
for handle, dag_run_id in RUN_HANDLES.items():
    print(f"  {handle:<16} {dag_run_id or '(not run)'}")

print()
print("=" * 60)
print(f"  PASSED {passed}   FAILED {failed}   ERROR {errored}   "
      f"MANUAL {manual}   SKIPPED {skipped}")
if failed == 0 and errored == 0:
    print("  REGRESSION PASSED")
else:
    print("  REGRESSION FAILED")
print("=" * 60)
if not SPARK_AVAILABLE:
    print("  WARNING: no PySpark kernel, so no tracking-table assertion ran. This result "
          "cannot be used to sign off a regression — re-run on a PySpark kernel.")
if manual or skipped:
    print(f"  Note: {manual} manual and {skipped} skipped check(s) were not verified "
          f"automatically — see the list above before signing off.")

---
## Re-running this notebook after a DAG code change

Run top to bottom. The suite is idempotent: Step 4 undoes the reversible fault injection, and
every trigger uses a fresh `dag_run_id` derived from `REGRESSION_RUN_TAG`.

### What needs re-seeding, and when

**Nothing to re-seed — just re-run this notebook:**

- Any change to `migration_dag_mapr_to_s3.py`, `migration_dag_iceberg.py`, or
  `migrator_utils/`. The source fixtures are untouched by a code change.
- Re-running after a P-02 pass. The `chmod 000` is restored to 755 in Step 5f and again in
  Step 4 on the next run.
- Re-running after a P-06 pass. The corrupt Excel object is deleted in Step 4.
- Adding or changing an expectation in the Step 3 table.

**Re-seed with `setup-test-data.sh` before the next run:**

```bash
docker exec -u root mapr-edge-node bash /setup-test-data.sh
```

- **After any P-01 pass.** `hdfs dfs -rm -r` on `migration_db.db/products` is not reversible
  from this notebook, and `R-02` expects that table `VALIDATED` in the baseline pass. Step 4
  probes the path and warns when it is missing, but it cannot restore it.
- **After any P-03 or P-04 pass.** The corrupted Parquet file lives at the S3 *destination*,
  not the source, so DAG 1's next run re-copies a clean file — but only if DAG 1 actually
  re-copies rather than treating the destination as already current. Re-seeding the source and
  clearing the destination prefix is the reliable reset.
- **If the `[10/11]` row or partition counts change** in `setup-test-data.sh`. Re-seed, then
  update the `REGRESSION` rows in Step 3 to match — the script is the source of truth, not
  this notebook.

**Re-seed with `setup-distcp-test-data.sh` before the next run:**

```bash
docker exec -u root mapr-edge-node bash /setup-distcp-test-data.sh
```

- **If any `DISTCP-SIZING` row reports 0 bytes / 0 files**, or its `-m` / `-bandwidth` is
  unexpectedly `1 / 800`. That is the signature the script itself warns about: the fixtures
  stopped being `EXTERNAL`, so a `DROP TABLE` deleted the loaded data.
- **Whenever the sizing knobs change.** The Airflow Variables
  (`migration_distcp_target_bytes_per_mapper`, `..._min_mappers`, `..._max_mappers`,
  `..._target_aggregate_mbps`) and the Step 0 values must agree. The Step 3 expectations are
  computed from the Step 0 values by the `size_distcp_job` port, so a mismatch between Step 0
  and Airflow shows up as every sizing row failing at once.

### What still needs a human

Listed with reasons in Step 5g: **T-01** (YARN container memory), **P-07** (MapR ticket
expiry), **P-03** (needs a SNAPSHOT Excel row, mutually exclusive with `P-05b`), and
**TC-32** (needs a pre-fix DDL capture to diff against). The report marks these `MANUAL` or
`SKIPPED-BY-CONFIG` and never counts them as passes — check the summary's closing note before
signing off a release on this suite.